<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/14_decision_analysis_FULL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB14_FULL — Análise de decisão por célula

Pipeline PPCOMP/DM — ramo `_FULL` · Borg 2019 completo por célula.

## 1. Papel do notebook no pipeline

O `NB14_FULL` transforma os escores preditivos já produzidos no ramo `_FULL` em análise operacional de decisão sob incerteza. Ele não reabre o treinamento dos modelos do `NB11_FULL`, nem a comparação LSTM do `NB13_FULL`/`NB13a_FULL`; sua função é consumir os escores disponíveis, medir antecipabilidade por episódio e varrer limiares `tau` para diferentes relações de custo entre falso positivo e falso negativo.

A unidade experimental continua sendo a célula Borg (`cell_id`). Cada célula é processada como réplica independente e somente depois os resultados são consolidados em `aggregate`. Não há concatenação temporal das células para cálculo de métricas, episódios, custos ou lead time.

## 2. Pergunta respondida

> Dado o escore de risco temporal do modelo vencedor por célula, quais limiares `tau` oferecem melhor compromisso entre falsos positivos e falsos negativos, e qual fração dos episódios críticos é antecipável dentro do horizonte herdado `H`?

A etapa responde essa pergunta em três níveis:

1. **por janela pontuada** — precisão, recall, F1, FPR, FNR, falsos alertas por dia e custo `C(tau)`;
2. **por episódio crítico** — episódios avaliáveis, episódios antecipados, taxa de antecipação e lead time;
3. **por célula e aggregate** — variação entre células, identificação de células com suporte fraco e consolidação citável para o `NB15_FULL` e para a dissertação.

## 3. Governança da fonte de escore

O `NB14_FULL` usa `NB11_FULL` como fonte primária conservadora. A LSTM de `NB13_FULL` ou `NB13a_FULL` só pode substituir o `NB11_FULL` se houver recomendação governada explícita no campo `nb14_score_source_recommendation = lstm_primary_for_nb14`. O campo bruto `decision` e textos livres de justificativa não são autoritativos para promover LSTM, porque podem refletir ganho nominal não robusto.

Na execução _FULL atual, o comportamento esperado é manter `NB11_FULL` como fonte primária em todas as células e tratar LSTM apenas como sensibilidade quando seus escores estiverem disponíveis. O `pilot_validation_summary` inclui uma trava `score_source_governance_conservative` para sinalizar qualquer promoção inesperada de LSTM antes do `NB15_FULL`.

## 4. Entradas esperadas

### Artefatos obrigatórios do `NB10_FULL` por célula

- `04-reports/99_FULL_downstream/10_FULL_threshold_diagnostics/cell_<id>/10_FULL_scenario_episodes_cell_<id>.parquet`
- `04-reports/99_FULL_downstream/10_FULL_threshold_diagnostics/cell_<id>/10_FULL_scenario_series_states_cell_<id>.parquet`
- `04-reports/99_FULL_downstream/10_FULL_threshold_diagnostics/cell_<id>/10_FULL_scenario_decisions_cell_<id>.csv`, quando disponível

Esses artefatos definem episódios críticos, estados operacionais, cenário, horizonte `H`, granularidade temporal e corte herdado. O `NB14_FULL` não recalcula episódios nem limiar de criticidade.

### Artefatos esperados do `NB11_FULL`

- `04-reports/99_FULL_downstream/11_FULL_model_baselines/aggregate/11_FULL_winner_model.json`, se disponível;
- `04-reports/99_FULL_downstream/11_FULL_model_baselines/aggregate/11_FULL_metrics_summary_by_cell.csv`, se disponível;
- escores por célula ou allcells, preferencialmente calibrados, por exemplo:
  - `11_FULL_scores_calibrated_cell_<id>.parquet`;
  - `11_FULL_scores_raw_cell_<id>.parquet`;
  - `11_FULL_scores_calibrated_allcells.parquet`;
  - `11_FULL_scores_raw_allcells.parquet`.

Quando a calibração existir, os escores seguem sendo interpretados de forma conservadora como **escores relativos de risco temporal**, não como probabilidades operacionais plenamente acionáveis sem validação adicional.

### Artefatos opcionais do `NB12_FULL`, `NB13_FULL` e `NB13a_FULL`

- `12_FULL_*score*.parquet`, para sensibilidade `TRAIN_P95` ou outro cenário de robustez;
- `13_FULL_lstm_scores_cell_<id>.parquet`, se o `NB13_FULL` tiver autorizado uso da LSTM;
- `13a_FULL_lstm_tuning_decision_to_nb14_cell_<id>.json` e `13a_FULL_lstm_tuning_scores_cell_<id>.parquet`, se o tuning compacto tiver sido executado.

Contrato de integração com o `NB13a_FULL`:

```text
NB14_FULL deve consumir nb14_score_source_recommendation, não apenas decision.
```

Assim, a LSTM só se torna fonte primária se o campo soberano recomendar explicitamente `lstm_primary_for_nb14`. Caso contrário, o `NB11_FULL` permanece a fonte primária, e a LSTM pode aparecer apenas como sensibilidade opcional.

## 5. Saídas geradas

As saídas são gravadas em:

```text
04-reports/99_FULL_downstream/14_FULL_decision_analysis/
  cell_a/
  ...
  cell_h/
  aggregate/
```

Por célula, o notebook gera:

- `14_FULL_scores_selected_cell_<id>.parquet`;
- `14_FULL_threshold_metrics_by_tau_cell_<id>.csv`;
- `14_FULL_cost_curve_cell_<id>.csv`;
- `14_FULL_optimal_tau_by_cost_cell_<id>.csv`;
- `14_FULL_false_alerts_by_tau_cell_<id>.csv`;
- `14_FULL_episode_anticipability_by_tau_cell_<id>.csv`;
- `14_FULL_episode_anticipability_cell_<id>.csv`;
- `14_FULL_episode_duration_summary_cell_<id>.csv`;
- `14_FULL_scenario_summary_cell_<id>.csv`;
- `14_FULL_decision_analysis_summary_cell_<id>.json`;
- figuras diagnósticas em `cell_<id>/figures/`.

No aggregate, gera:

- `14_FULL_scenario_summary_by_cell.csv`;
- `14_FULL_cost_summary_by_cell.csv`;
- `14_FULL_optimal_tau_by_cost_allcells.csv`;
- `14_FULL_threshold_metrics_by_tau_allcells.csv`;
- `14_FULL_cost_curve_allcells.csv`;
- `14_FULL_episode_anticipability_by_tau_allcells.csv`;
- `14_FULL_episode_anticipability_allcells.csv`;
- `14_FULL_episode_duration_summary_allcells.csv`;
- `14_FULL_nb14_summary.json`;
- `14_FULL_delta_vs_canonical.csv`;
- `14_FULL_pilot_validation_summary.csv`;
- `14_FULL_artifact_manifest_sha256.csv`;
- figuras consolidadas em `aggregate/figures/`.

## 6. Travas metodológicas

- O notebook lê apenas artefatos do ramo `_FULL` em `99_FULL_downstream`.
- Nenhum arquivo é escrito em `03-features`, `04-reports/` canônico ou diretórios do Kaggle canônico.
- O corte treino/teste e os cenários são herdados dos notebooks upstream; o `NB14_FULL` não recalcula limiar, split nem episódios.
- Todos os artefatos de saída possuem prefixo obrigatório `14_FULL_`.
- Células sem escores ou episódios suficientes são registradas como falha/skip com justificativa, sem contaminação das demais células.

## 7. Diferenças em relação ao NB14 canônico

1. Entrada segmentada por `cell_id`, a partir de `10_FULL_*`, `11_FULL_*`, `12_FULL_*`, `13_FULL_*` e `13a_FULL_*`.
2. Fonte primária de escores decidida por célula, com `NB11_FULL` como padrão conservador.
3. Uso do campo `nb14_score_source_recommendation` do `NB13a_FULL` quando existir.
4. Saídas em `99_FULL_downstream/14_FULL_decision_analysis/cell_<id>` e `aggregate`.
5. Figuras e CSVs por célula, mais sínteses allcells.
6. Métricas e custos calculados por célula, nunca sobre série concatenada.
7. Manifesto SHA-256 e delta CSV próprios do ramo `_FULL`.


In [ ]:

# ============================================================
# NB14_FULL — Episódios, Antecipabilidade e Função de Custo C(tau) por célula
# Pipeline PPCOMP/DM — ramo _FULL
# ============================================================
# Versão proposta:
# 1. Iteração por cell_id, sem concatenação temporal para cálculo de métricas.
# 2. Consumo de episódios/estados do NB10_FULL.
# 3. Consumo preferencial de escores do NB11_FULL por célula.
# 4. Uso de NB13_FULL/NB13a_FULL somente se a decisão upstream permitir.
# 5. Varredura tau em [0,1], função C(tau) e antecipabilidade por episódio.
# 6. Saídas 14_FULL_* em 99_FULL_downstream/14_FULL_decision_analysis.
# ============================================================

import os
import re
import json
import math
import hashlib
import warnings
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 240)

# ============================================================
# 0. Utilitários gerais
# ============================================================

RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
RUN_TIMESTAMP_FILE = datetime.now().strftime("%Y%m%d_%H%M%S")
NB14_FULL_VERSION = "FULL_DECISION_ANALYSIS_BY_CELL_GOVERNED_V2"


def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def log(msg):
    print(f"[{now_str()}] {msg}", flush=True)


def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def json_ready(obj):
    if isinstance(obj, dict):
        return {str(k): json_ready(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_ready(v) for v in obj]
    if isinstance(obj, tuple):
        return [json_ready(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if not np.isfinite(obj) else float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if obj is pd.NA:
        return None
    if isinstance(obj, float) and not math.isfinite(obj):
        return None
    return obj


def read_json(path, default=None, required=False, label="json"):
    if path is None:
        if required:
            raise FileNotFoundError(f"{label}: caminho None")
        return default
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(f"{label} não encontrado: {path}")
        return default
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        if required:
            raise
        return default


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(json_ready(obj), f, ensure_ascii=False, indent=2)
    return path


def save_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if df is None:
        df = pd.DataFrame()
    df.to_csv(path, index=False)
    return path


def save_parquet(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if df is None:
        df = pd.DataFrame()
    df.to_parquet(path, index=False)
    return path


def safe_div(num, den):
    try:
        den = float(den)
        if den == 0 or pd.isna(den):
            return np.nan
        return float(num) / den
    except Exception:
        return np.nan


def file_sha256(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def build_artifact_manifest(root_dir, output_csv):
    root_dir = Path(root_dir)
    rows = []
    for p in sorted(root_dir.rglob("*")):
        if not p.is_file():
            continue
        if p.name == Path(output_csv).name:
            continue
        try:
            rows.append({
                "relative_path": str(p.relative_to(root_dir)),
                "path": str(p),
                "size_bytes": int(p.stat().st_size),
                "sha256": file_sha256(p),
                "mtime": datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec="seconds"),
            })
        except Exception as exc:
            rows.append({
                "relative_path": str(p),
                "path": str(p),
                "size_bytes": np.nan,
                "sha256": None,
                "mtime": None,
                "error": repr(exc),
            })
    df = pd.DataFrame(rows)
    save_csv(df, output_csv)
    return df


def list_dir_safe(path, max_items=80):
    path = Path(path)
    if not path.exists():
        return []
    out = []
    for p in sorted(path.iterdir())[:max_items]:
        out.append({"name": p.name, "is_dir": p.is_dir(), "size": p.stat().st_size if p.is_file() else None})
    return out


# ============================================================
# 1. Bootstrap, caminhos oficiais _FULL e travas de herança
# ============================================================

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        log(f"[AVISO] Não foi possível montar Google Drive automaticamente: {exc}")

BASE_DIR = (
    Path("/content/drive/MyDrive/Mestrado")
    if Path("/content/drive/MyDrive/Mestrado").exists()
    else Path.cwd()
)

FULL_REPORTS_DIR = ensure_dir(BASE_DIR / "04-reports" / "99_FULL_downstream")
STAGE10_DIR = FULL_REPORTS_DIR / "10_FULL_threshold_diagnostics"
STAGE11_DIR = FULL_REPORTS_DIR / "11_FULL_model_baselines"
STAGE11_AGG_DIR = STAGE11_DIR / "aggregate"
STAGE12_DIR = FULL_REPORTS_DIR / "12_FULL_sensitivity"
STAGE12_ALT_DIR = FULL_REPORTS_DIR / "12_FULL_sensitivity_diagnostics"
STAGE13_DIR = FULL_REPORTS_DIR / "13_FULL_lstm"
STAGE13A_DIR = FULL_REPORTS_DIR / "13a_FULL_lstm_tuning"
STAGE14_DIR = ensure_dir(FULL_REPORTS_DIR / "14_FULL_decision_analysis")
AGGREGATE_DIR = ensure_dir(STAGE14_DIR / "aggregate")
AGG_FIGURES_DIR = ensure_dir(AGGREGATE_DIR / "figures")

FULL_CELLS = list("abcdefgh")

# Escopo de execução governado. Pode ser sobrescrito no Colab antes da célula:
# EXECUTION_SCOPE = "pilot_a" ou "allcells" ou "custom" + ACTIVE_CELLS = ["a", "g"]
if "EXECUTION_SCOPE" not in globals():
    EXECUTION_SCOPE = os.environ.get("NB14_FULL_EXECUTION_SCOPE", "allcells")
if "ACTIVE_CELLS" not in globals():
    env_cells = os.environ.get("NB14_FULL_ACTIVE_CELLS", "").strip()
    if EXECUTION_SCOPE == "pilot_a":
        ACTIVE_CELLS = ["a"]
    elif EXECUTION_SCOPE == "custom" and env_cells:
        ACTIVE_CELLS = [c.strip().lower() for c in env_cells.split(",") if c.strip()]
    elif EXECUTION_SCOPE == "custom":
        ACTIVE_CELLS = ["a"]
    else:
        ACTIVE_CELLS = FULL_CELLS[:]

ACTIVE_CELLS = [str(c).lower().replace("cell_", "") for c in ACTIVE_CELLS]
assert set(ACTIVE_CELLS).issubset(set(FULL_CELLS)), f"ACTIVE_CELLS inválido: {ACTIVE_CELLS}"
assert len(ACTIVE_CELLS) > 0, "ACTIVE_CELLS vazio."

CONFIG = {
    "main_scenario_default": "W5_K24_H12_P1_TRAIN_M2S",
    "include_nb12_sensitivity_scores": True,
    "include_lstm_optional_sensitivity_scores": True,
    "p95_scenario_default": "W5_K24_H12_P1_TRAIN_P95",
    "tau_step": 0.01,
    "classification_reference_tau": 0.50,
    "cost_ratios": [
        {"cost_label": "cFP1_cFN5", "c_fp": 1.0, "c_fn": 5.0},
        {"cost_label": "cFP1_cFN10", "c_fp": 1.0, "c_fn": 10.0},
        {"cost_label": "cFP1_cFN20", "c_fp": 1.0, "c_fn": 20.0},
    ],
    "low_count_threshold_duration_bin": 5,
    "prefer_fixed_protocol_scores": True,
    "prefer_calibrated_nb11_scores": True,
    "fail_fast": False,
    "save_figures": True,
    "timeline_context_h_multiplier": 3,
}

TAU_GRID = np.round(np.arange(0.0, 1.0 + CONFIG["tau_step"] / 2, CONFIG["tau_step"]), 4)

# Travas _FULL: caminhos e prefixos.
assert "99_FULL_downstream" in str(FULL_REPORTS_DIR), f"FULL_REPORTS_DIR inesperado: {FULL_REPORTS_DIR}"
assert "03-features" not in str(STAGE14_DIR), "NB14_FULL não pode gravar em 03-features."
assert STAGE14_DIR.name == "14_FULL_decision_analysis", f"Nome de pasta inesperado: {STAGE14_DIR}"
assert all(str(c) in FULL_CELLS for c in ACTIVE_CELLS)

log("=" * 110)
log("NB14_FULL — Análise de decisão por célula")
log(f"NB14_FULL_VERSION = {NB14_FULL_VERSION}")
log(f"RUN_TIMESTAMP = {RUN_TIMESTAMP}")
log(f"BASE_DIR = {BASE_DIR}")
log(f"FULL_REPORTS_DIR = {FULL_REPORTS_DIR}")
log(f"STAGE10_DIR = {STAGE10_DIR}")
log(f"STAGE11_DIR = {STAGE11_DIR}")
log(f"STAGE12_DIR = {STAGE12_DIR}")
log(f"STAGE13_DIR = {STAGE13_DIR}")
log(f"STAGE13A_DIR = {STAGE13A_DIR}")
log(f"STAGE14_DIR = {STAGE14_DIR}")
log(f"EXECUTION_SCOPE = {EXECUTION_SCOPE}")
log(f"ACTIVE_CELLS = {ACTIVE_CELLS}")
log("=" * 110)


# ============================================================
# 2. Inferência de colunas, cenários e localização de artefatos
# ============================================================


def infer_column(df, candidates, required=True, label="coluna"):
    cols = list(df.columns)
    lower_map = {str(c).lower(): c for c in cols}
    normalized_map = {re.sub(r"[^a-z0-9]+", "", str(c).lower()): c for c in cols}
    for c in candidates:
        if c in df.columns:
            return c
        cl = str(c).lower()
        if cl in lower_map:
            return lower_map[cl]
        cn = re.sub(r"[^a-z0-9]+", "", cl)
        if cn in normalized_map:
            return normalized_map[cn]
    if required:
        raise ValueError(
            f"Não foi possível inferir {label}. Candidatos: {candidates}. Colunas disponíveis: {cols}"
        )
    return None


def parse_scenario_label(label):
    label = str(label)
    # Usa re.search, não re.match, para tolerar rótulos prefixados por cell_<id>_.
    # O threshold é limitado a caracteres nominais para evitar capturas livres por ".+".
    m = re.search(
        r"W(?P<W>\d+)_K(?P<K>\d+)_H(?P<H>\d+)_P(?P<P>\d+)_(?P<threshold>[A-Za-z0-9_]+)",
        label,
    )
    if not m:
        return {
            "scenario_label": label,
            "window_minutes": np.nan,
            "k_windows": np.nan,
            "h_windows": np.nan,
            "persistence_min_windows": np.nan,
            "threshold_family": "unknown",
        }
    d = m.groupdict()
    return {
        "scenario_label": label,
        "window_minutes": int(d["W"]),
        "k_windows": int(d["K"]),
        "h_windows": int(d["H"]),
        "persistence_min_windows": int(d["P"]),
        "threshold_family": str(d["threshold"]).lower(),
    }


def get_window_minutes(label, fallback=5):
    try:
        v = parse_scenario_label(label).get("window_minutes")
        return int(v) if pd.notna(v) else int(fallback)
    except Exception:
        return int(fallback)


def get_h_windows(label, fallback=12):
    try:
        v = parse_scenario_label(label).get("h_windows")
        return int(v) if pd.notna(v) else int(fallback)
    except Exception:
        return int(fallback)


def duration_bucket_from_windows(duration_windows):
    if pd.isna(duration_windows):
        return "unknown"
    d = int(duration_windows)
    if d <= 1:
        return "1 janela"
    if 2 <= d <= 5:
        return "2–5 janelas"
    if 6 <= d <= 20:
        return "6–20 janelas"
    return ">20 janelas"


def duration_bucket_order(bucket):
    order = {"1 janela": 1, "2–5 janelas": 2, "6–20 janelas": 3, ">20 janelas": 4, "unknown": 99}
    return order.get(str(bucket), 99)


def find_existing(candidates, required=False, label="arquivo"):
    for p in candidates:
        if p is None:
            continue
        p = Path(p)
        if p.exists():
            return p
    if required:
        msg = f"{label} obrigatório não encontrado. Candidatos:\n" + "\n".join(str(p) for p in candidates)
        raise FileNotFoundError(msg)
    return None


def sorted_existing_globs(roots, patterns):
    hits = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pattern in patterns:
            hits.extend(root.glob(pattern))
    hits = [p for p in hits if p.exists() and p.is_file()]
    return sorted(set(hits), key=lambda p: (0 if "aggregate" not in p.parts else 1, str(p)))


def read_csv_if_exists(path):
    path = Path(path) if path else None
    if path is None or not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def read_parquet_if_exists(path):
    path = Path(path) if path else None
    if path is None or not path.exists():
        return pd.DataFrame()
    return pd.read_parquet(path)


# ============================================================
# 3. Contexto NB11/NB13/NB13a e decisão de fonte de escores
# ============================================================

PATH_NB11_WINNER = find_existing([
    STAGE11_AGG_DIR / "11_FULL_winner_model.json",
    STAGE11_AGG_DIR / "11_FULL_winner_by_cell.json",
    STAGE11_DIR / "11_FULL_winner_model.json",
], required=False, label="winner NB11_FULL")

PATH_NB11_METRICS_BY_CELL = find_existing([
    STAGE11_AGG_DIR / "11_FULL_metrics_summary_by_cell.csv",
    STAGE11_AGG_DIR / "11_FULL_metrics_by_cell.csv",
    STAGE11_AGG_DIR / "11_FULL_model_metrics_by_cell.csv",
], required=False, label="métricas NB11_FULL por célula")

winner_global = read_json(PATH_NB11_WINNER, default={}, required=False, label="winner NB11_FULL")
df_nb11_metrics_by_cell = read_csv_if_exists(PATH_NB11_METRICS_BY_CELL)

log("Artefatos de referência NB11_FULL:")
log(f"PATH_NB11_WINNER = {PATH_NB11_WINNER}")
log(f"PATH_NB11_METRICS_BY_CELL = {PATH_NB11_METRICS_BY_CELL}")
if not df_nb11_metrics_by_cell.empty:
    log(f"df_nb11_metrics_by_cell: {df_nb11_metrics_by_cell.shape}")


def select_nb11_context_for_cell(cell_id):
    """Resolve cenário/modelo/feature_set vencedor do NB11_FULL para a célula.

    A função é tolerante a pequenas variações de schema para permitir execução
    em artefatos gerados em rodadas anteriores.
    """
    ctx = {
        "cell_id": cell_id,
        "scenario_label": CONFIG["main_scenario_default"],
        "model": "unknown",
        "feature_set": "unknown",
        "f1_tscv_mean": np.nan,
        "source": "fallback_default",
    }

    if not df_nb11_metrics_by_cell.empty and "cell_id" in df_nb11_metrics_by_cell.columns:
        rows = df_nb11_metrics_by_cell[df_nb11_metrics_by_cell["cell_id"].astype(str).str.lower().eq(str(cell_id).lower())].copy()
        if not rows.empty:
            # Se houver coluna de vencedor, prioriza linhas vencedoras; senão escolhe maior F1.
            winner_cols = [c for c in rows.columns if str(c).lower() in {"is_winner", "winner", "selected", "is_selected"}]
            if winner_cols:
                mask = rows[winner_cols[0]].astype(str).str.lower().isin(["1", "true", "yes", "sim", "winner", "selected"])
                if mask.any():
                    rows = rows[mask].copy()
            f1_cols = [c for c in ["f1_tscv_mean", "f1_mean", "mean_f1", "f1"] if c in rows.columns]
            if f1_cols:
                rows["__f1_sort__"] = pd.to_numeric(rows[f1_cols[0]], errors="coerce")
                rows = rows.sort_values("__f1_sort__", ascending=False)
            row = rows.iloc[0].to_dict()
            scenario_col = infer_column(pd.DataFrame([row]), ["scenario_label", "scenario", "config_label"], required=False)
            model_col = infer_column(pd.DataFrame([row]), ["model", "model_name", "estimator"], required=False)
            feature_col = infer_column(pd.DataFrame([row]), ["feature_set", "featureset", "feature_group"], required=False)
            ctx.update({
                "scenario_label": str(row.get(scenario_col, ctx["scenario_label"])) if scenario_col else ctx["scenario_label"],
                "model": str(row.get(model_col, ctx["model"])) if model_col else ctx["model"],
                "feature_set": str(row.get(feature_col, ctx["feature_set"])) if feature_col else ctx["feature_set"],
                "source": str(PATH_NB11_METRICS_BY_CELL),
            })
            if f1_cols:
                ctx["f1_tscv_mean"] = float(pd.to_numeric(pd.Series([row.get(f1_cols[0])]), errors="coerce").iloc[0])
            return ctx

    # Fallback para JSON global ou por célula.
    if isinstance(winner_global, dict) and winner_global:
        # Alguns JSONs são {cell_id: {...}}.
        cell_payload = None
        for key in [cell_id, f"cell_{cell_id}"]:
            if isinstance(winner_global.get(key), dict):
                cell_payload = winner_global.get(key)
                break
        if cell_payload is None:
            for key in ["winner", "selected", "best", "global_winner"]:
                if isinstance(winner_global.get(key), dict):
                    cell_payload = winner_global.get(key)
                    break
        if cell_payload is None:
            cell_payload = winner_global
        ctx.update({
            "scenario_label": str(cell_payload.get("scenario_label", cell_payload.get("scenario", ctx["scenario_label"]))),
            "model": str(cell_payload.get("model", cell_payload.get("model_name", ctx["model"]))),
            "feature_set": str(cell_payload.get("feature_set", cell_payload.get("featureset", ctx["feature_set"]))),
            "f1_tscv_mean": cell_payload.get("f1_tscv_mean", cell_payload.get("f1_mean", ctx["f1_tscv_mean"])),
            "source": str(PATH_NB11_WINNER),
        })
        try:
            ctx["f1_tscv_mean"] = float(ctx["f1_tscv_mean"])
        except Exception:
            ctx["f1_tscv_mean"] = np.nan
    return ctx


def load_nb13a_decision(cell_id):
    candidates = [
        STAGE13A_DIR / f"cell_{cell_id}" / f"13a_FULL_lstm_tuning_decision_to_nb14_cell_{cell_id}.json",
        STAGE13A_DIR / f"cell_{cell_id}" / "13a_FULL_lstm_tuning_decision_to_nb14.json",
    ]
    path = find_existing(candidates, required=False, label=f"decisão NB13a_FULL cell_{cell_id}")
    if path:
        payload = read_json(path, default={}, required=False)
        if isinstance(payload, dict):
            payload["_path"] = str(path)
            return payload

    # Fallback no summary aggregate do NB13a.
    summary_path = STAGE13A_DIR / "aggregate" / "13a_FULL_lstm_tuning_summary.json"
    summary = read_json(summary_path, default={}, required=False)
    for item in summary.get("decisions_by_cell", []) if isinstance(summary, dict) else []:
        if str(item.get("cell_id", "")).lower() == str(cell_id).lower():
            item = dict(item)
            item["_path"] = str(summary_path)
            return item
    return {}


def load_nb13_decision(cell_id):
    candidates = [
        STAGE13_DIR / f"cell_{cell_id}" / f"13_FULL_lstm_decision_to_nb14_cell_{cell_id}.json",
        STAGE13_DIR / f"cell_{cell_id}" / f"13_FULL_lstm_summary_cell_{cell_id}.json",
        STAGE13_DIR / f"cell_{cell_id}" / "13_lstm_summary.json",
    ]
    path = find_existing(candidates, required=False, label=f"decisão NB13_FULL cell_{cell_id}")
    if path:
        payload = read_json(path, default={}, required=False)
        if isinstance(payload, dict):
            payload["_path"] = str(path)
            return payload
    return {}


def recommend_primary_score_family(cell_id):
    """Decide família primária de escores para o NB14_FULL.

    Regra de governança:
    - NB13a_FULL, quando presente, é autoritativo somente pelo campo
      nb14_score_source_recommendation.
    - NB13_FULL, em fallback, também é autoritativo somente pelo campo
      nb14_score_source_recommendation.
    - Campos livres como decision, notes, rationale ou valores textuais contendo
      lstm_scores_can_feed_nb14 NÃO promovem LSTM para fonte primária.
    - Qualquer ausência, valor desconhecido ou campo fora do contrato cai no padrão
      conservador NB11_FULL.

    Retorno: nb11, nb13, nb13a.
    """
    allowed_nb11 = {"nb11_primary", "nb11_primary_lstm_optional_sensitivity"}

    nb13a = load_nb13a_decision(cell_id)
    rec13a = str(nb13a.get("nb14_score_source_recommendation", "")).strip().lower() if isinstance(nb13a, dict) else ""
    if rec13a == "lstm_primary_for_nb14":
        return "nb13a", "NB13a_FULL (campo governado) recomendou lstm_primary_for_nb14", nb13a
    if rec13a in allowed_nb11:
        return "nb11", f"NB13a_FULL (campo governado) recomendou {rec13a}", nb13a
    if nb13a:
        return (
            "nb11",
            "Fonte primária conservadora: NB11_FULL; NB13a_FULL existe, mas não trouxe recomendação governada autoritativa",
            nb13a,
        )

    nb13 = load_nb13_decision(cell_id)
    rec13 = str(nb13.get("nb14_score_source_recommendation", "")).strip().lower() if isinstance(nb13, dict) else ""
    if rec13 == "lstm_primary_for_nb14":
        return "nb13", "NB13_FULL (campo governado) recomendou lstm_primary_for_nb14", nb13
    if rec13 in allowed_nb11:
        return "nb11", f"NB13_FULL (campo governado) recomendou {rec13}", nb13
    return (
        "nb11",
        "Fonte primária conservadora: NB11_FULL; decision bruto do NB13 é não-autoritativo",
        nb13,
    )


# ============================================================
# 4. Localização de episódios, estados e escores
# ============================================================


def resolve_stage10_artifacts(cell_id):
    cell_dir = STAGE10_DIR / f"cell_{cell_id}"
    episodes = find_existing([
        cell_dir / f"10_FULL_scenario_episodes_cell_{cell_id}.parquet",
        cell_dir / "10_FULL_scenario_episodes.parquet",
        cell_dir / "10_scenario_episodes.parquet",
    ], required=False, label=f"episódios NB10_FULL cell_{cell_id}")
    if episodes is None:
        hits = sorted_existing_globs([cell_dir], [f"*episodes*cell_{cell_id}*.parquet", "*episodes*.parquet"])
        episodes = hits[0] if hits else None

    states = find_existing([
        cell_dir / f"10_FULL_scenario_series_states_cell_{cell_id}.parquet",
        cell_dir / "10_FULL_scenario_series_states.parquet",
        cell_dir / "10_scenario_series_states.parquet",
    ], required=False, label=f"estados NB10_FULL cell_{cell_id}")
    if states is None:
        hits = sorted_existing_globs([cell_dir], [f"*series*states*cell_{cell_id}*.parquet", "*states*.parquet"])
        states = hits[0] if hits else None

    decisions = find_existing([
        cell_dir / f"10_FULL_scenario_decisions_cell_{cell_id}.csv",
        cell_dir / "10_FULL_scenario_decisions.csv",
        cell_dir / "10_scenario_decisions.csv",
    ], required=False, label=f"decisões NB10_FULL cell_{cell_id}")
    if decisions is None:
        hits = sorted_existing_globs([cell_dir], [f"*decisions*cell_{cell_id}*.csv", "*decisions*.csv"])
        decisions = hits[0] if hits else None

    return {"episodes": episodes, "states": states, "decisions": decisions, "cell_dir": cell_dir}


def score_exact_candidates(stage_dir, aggregate_dir, family, cell_id):
    """Candidatos explícitos de escores por família."""
    cell_dir = stage_dir / f"cell_{cell_id}"
    if family == "nb11":
        names_cell = [
            f"11_FULL_scores_calibrated_cell_{cell_id}.parquet",
            f"11_FULL_scores_fixed_cell_{cell_id}.parquet",
            f"11_FULL_scores_cell_{cell_id}.parquet",
            f"11_FULL_scores_raw_cell_{cell_id}.parquet",
            f"11_FULL_model_scores_cell_{cell_id}.parquet",
            f"11_FULL_prediction_scores_cell_{cell_id}.parquet",
            "11_FULL_scores_calibrated.parquet",
            "11_FULL_scores_raw.parquet",
            "11_scores_calibrated.parquet",
            "11_scores_raw.parquet",
        ]
        names_agg = [
            "11_FULL_scores_calibrated_allcells.parquet",
            "11_FULL_scores_raw_allcells.parquet",
            "11_FULL_scores_allcells.parquet",
            "11_FULL_model_scores_allcells.parquet",
            "11_FULL_prediction_scores_allcells.parquet",
        ]
    elif family == "nb12":
        names_cell = [
            f"12_FULL_scores_sensitivity_fixed_cell_{cell_id}.parquet",
            f"12_FULL_scores_cell_{cell_id}.parquet",
            f"12_FULL_sensitivity_scores_cell_{cell_id}.parquet",
            "12_scores_sensitivity_fixed.parquet",
        ]
        names_agg = [
            "12_FULL_scores_sensitivity_fixed_allcells.parquet",
            "12_FULL_scores_allcells.parquet",
            "12_FULL_sensitivity_scores_allcells.parquet",
        ]
    elif family == "nb13":
        names_cell = [
            f"13_FULL_lstm_scores_cell_{cell_id}.parquet",
            f"13_FULL_lstm_fixed_scores_cell_{cell_id}.parquet",
            f"13_FULL_scores_cell_{cell_id}.parquet",
            "13_lstm_scores.parquet",
        ]
        names_agg = [
            "13_FULL_lstm_scores_allcells.parquet",
            "13_FULL_scores_allcells.parquet",
        ]
    elif family == "nb13a":
        names_cell = [
            f"13a_FULL_lstm_tuning_scores_cell_{cell_id}.parquet",
            f"13a_FULL_scores_cell_{cell_id}.parquet",
        ]
        names_agg = [
            "13a_FULL_lstm_tuning_scores_allcells.parquet",
            "13a_FULL_scores_allcells.parquet",
        ]
    else:
        names_cell, names_agg = [], []
    return [cell_dir / n for n in names_cell] + [aggregate_dir / n for n in names_agg]


def resolve_score_file_for_family(cell_id, family):
    family = str(family).lower()
    if family == "nb11":
        stage_dir, agg_dir = STAGE11_DIR, STAGE11_AGG_DIR
    elif family == "nb12":
        stage_dir = STAGE12_DIR if STAGE12_DIR.exists() else STAGE12_ALT_DIR
        agg_dir = stage_dir / "aggregate"
    elif family == "nb13":
        stage_dir, agg_dir = STAGE13_DIR, STAGE13_DIR / "aggregate"
    elif family == "nb13a":
        stage_dir, agg_dir = STAGE13A_DIR, STAGE13A_DIR / "aggregate"
    else:
        raise ValueError(f"family inválida: {family}")

    exact = find_existing(score_exact_candidates(stage_dir, agg_dir, family, cell_id), required=False)
    if exact:
        return exact

    cell_dir = stage_dir / f"cell_{cell_id}"
    hits = sorted_existing_globs([cell_dir], [f"*score*cell_{cell_id}*.parquet", "*score*.parquet"])
    if hits:
        # Preferência por calibrado quando NB11; caso contrário, primeiro por nome estável.
        if family == "nb11" and CONFIG["prefer_calibrated_nb11_scores"]:
            cal = [p for p in hits if "calib" in p.name.lower()]
            if cal:
                return cal[0]
        return hits[0]

    # Fallback aggregate allcells.
    hits = sorted_existing_globs([agg_dir], ["*score*allcells*.parquet", "*score*.parquet"])
    if hits:
        if family == "nb11" and CONFIG["prefer_calibrated_nb11_scores"]:
            cal = [p for p in hits if "calib" in p.name.lower()]
            if cal:
                return cal[0]
        return hits[0]
    return None


# ============================================================
# 5. Padronização dos escores e episódios
# ============================================================


def standardize_score_frame(df, cell_id, source_name, source_path):
    if df is None or df.empty:
        raise ValueError(f"Score frame vazio: {source_name} {source_path}")
    df = df.copy()
    df = df.loc[:, ~df.columns.duplicated()].copy()

    cell_col = infer_column(df, ["cell_id", "cell", "borg_cell"], required=False, label="cell_id nos escores")
    if cell_col:
        df = df[df[cell_col].astype(str).str.lower().str.replace("cell_", "", regex=False).eq(str(cell_id).lower())].copy()
    if df.empty:
        raise ValueError(f"Score frame sem linhas para cell_{cell_id}: {source_path}")

    scenario_col = infer_column(df, ["scenario_label", "scenario", "scenario_id", "config_label"], label="cenário nos escores")
    time_col = infer_column(df, ["bucket_id", "time_order", "bucket", "row_end_bucket_id", "row_end_position", "bucket_end", "pos", "row_position"], label="coluna temporal nos escores")
    y_col = infer_column(df, ["y_true", "y", "target", "label", "target_within_horizon", "target_event_within_horizon", "y_test"], label="alvo nos escores")
    score_col = infer_column(df, [
        "score_calibrated", "y_score_calibrated", "calibrated_score", "score_platt", "score_isotonic",
        "score_raw", "y_score", "score", "probability", "prob", "proba", "pred_proba", "y_pred_proba",
        "positive_proba", "proba_1", "score_lstm", "y_score_lstm", "y_score_lstm_tuned", "score_lstm_tuned",
    ], label="escore")

    protocol_col = infer_column(df, ["protocol", "validation", "split_protocol", "validation_protocol", "evaluation_protocol"], required=False, label="protocolo")
    if CONFIG["prefer_fixed_protocol_scores"] and protocol_col:
        p = df[protocol_col].astype(str).str.lower()
        fixed_mask = p.str.contains("fixed|80_20|80/20|holdout|test|inherited_split", regex=True, na=False)
        if fixed_mask.sum() > 0:
            df = df[fixed_mask].copy()

    split_col = infer_column(df, ["split", "dataset_split"], required=False, label="split")
    if split_col:
        # Para escores de política operacional, preservar teste/fixed quando explícito.
        split_s = df[split_col].astype(str).str.lower()
        test_mask = split_s.isin(["test", "fixed_test", "holdout", "validation"])
        if test_mask.sum() > 0:
            df = df[test_mask].copy()

    model_col = infer_column(df, ["model", "model_name", "estimator"], required=False, label="modelo")
    feature_set_col = infer_column(df, ["feature_set", "featureset", "feature_group"], required=False, label="feature_set")
    fold_col = infer_column(df, ["fold"], required=False, label="fold")
    seed_col = infer_column(df, ["seed"], required=False, label="seed")
    window_col = infer_column(df, ["window_minutes", "W"], required=False, label="window_minutes")
    h_col = infer_column(df, ["h_windows", "H", "horizon_h"], required=False, label="h_windows")

    out = pd.DataFrame({
        "cell_id": str(cell_id),
        "scenario_label": df[scenario_col].astype(str).values,
        "bucket_id": pd.to_numeric(df[time_col], errors="coerce").values,
        "y_true": pd.to_numeric(df[y_col], errors="coerce").values,
        "score": pd.to_numeric(df[score_col], errors="coerce").values,
    })
    out["score_source"] = str(source_name)
    out["score_source_file"] = str(source_path)
    out["original_score_column"] = str(score_col)
    out["protocol"] = df[protocol_col].astype(str).values if protocol_col else "unknown"
    out["split"] = df[split_col].astype(str).values if split_col else "unknown"
    out["model"] = df[model_col].astype(str).values if model_col else "unknown"
    out["feature_set"] = df[feature_set_col].astype(str).values if feature_set_col else "unknown"
    out["fold"] = df[fold_col].values if fold_col else np.nan
    out["seed"] = df[seed_col].values if seed_col else np.nan
    out["window_minutes"] = pd.to_numeric(df[window_col], errors="coerce").values if window_col else np.nan
    out["h_windows"] = pd.to_numeric(df[h_col], errors="coerce").values if h_col else np.nan

    out = out.replace([np.inf, -np.inf], np.nan).dropna(subset=["scenario_label", "bucket_id", "y_true", "score"])
    out["bucket_id"] = out["bucket_id"].astype(int)
    out["y_true"] = out["y_true"].astype(int)
    out["score"] = out["score"].clip(0.0, 1.0)
    parsed = out["scenario_label"].map(parse_scenario_label).apply(pd.Series)
    parsed_window = pd.to_numeric(parsed.get("window_minutes", pd.Series(np.nan, index=out.index)), errors="coerce")
    parsed_h = pd.to_numeric(parsed.get("h_windows", pd.Series(np.nan, index=out.index)), errors="coerce")
    out["window_minutes"] = pd.to_numeric(out["window_minutes"], errors="coerce").fillna(parsed_window).fillna(5).astype(int)
    out["h_windows"] = pd.to_numeric(out["h_windows"], errors="coerce").fillna(parsed_h).fillna(12).astype(int)

    for c in parsed.columns:
        if c not in out.columns:
            out[c] = parsed[c]

    # Múltiplas seeds/folds/configurações para o mesmo bucket viram média de escore.
    group_cols = ["cell_id", "scenario_label", "bucket_id", "score_source"]
    agg = {
        "y_true": "max",
        "score": "mean",
        "score_source_file": "first",
        "original_score_column": "first",
        "protocol": lambda x: "|".join(sorted(set(map(str, x))))[:200],
        "split": lambda x: "|".join(sorted(set(map(str, x))))[:200],
        "model": lambda x: "|".join(sorted(set(map(str, x))))[:200],
        "feature_set": lambda x: "|".join(sorted(set(map(str, x))))[:200],
        "window_minutes": "first",
        "h_windows": "first",
        "k_windows": "first",
        "persistence_min_windows": "first",
        "threshold_family": "first",
    }
    out = out.groupby(group_cols, as_index=False).agg(agg)
    out["score_label"] = out["scenario_label"].astype(str) + " | " + out["score_source"].astype(str)
    return out.sort_values(["scenario_label", "score_source", "bucket_id"]).reset_index(drop=True)


def standardize_episodes(df, cell_id):
    if df is None or df.empty:
        raise ValueError(f"Episódios vazios para cell_{cell_id}")
    df = df.copy()
    cell_col = infer_column(df, ["cell_id", "cell", "borg_cell"], required=False, label="cell_id em episódios")
    if cell_col:
        df = df[df[cell_col].astype(str).str.lower().str.replace("cell_", "", regex=False).eq(str(cell_id).lower())].copy()
    scenario_col = infer_column(df, ["scenario_label", "scenario", "scenario_id", "config_label"], label="cenário em episódios")
    start_col = infer_column(df, ["episode_start_bucket_id", "start_bucket_id", "start_bucket", "bucket_start", "start", "start_bucket_id_inclusive"], label="início do episódio")
    end_col = infer_column(df, ["episode_end_bucket_id", "end_bucket_id", "end_bucket", "bucket_end", "end", "end_bucket_id_inclusive"], label="fim do episódio")
    duration_col = infer_column(df, ["duration_windows", "episode_duration_windows", "n_windows", "length_windows"], required=False, label="duração")

    out = pd.DataFrame({
        "cell_id": str(cell_id),
        "scenario_label": df[scenario_col].astype(str).values,
        "episode_start_bucket_id": pd.to_numeric(df[start_col], errors="coerce").values,
        "episode_end_bucket_id": pd.to_numeric(df[end_col], errors="coerce").values,
    })
    out = out.dropna(subset=["episode_start_bucket_id", "episode_end_bucket_id"])
    out["episode_start_bucket_id"] = out["episode_start_bucket_id"].astype(int)
    out["episode_end_bucket_id"] = out["episode_end_bucket_id"].astype(int)
    if duration_col:
        out["duration_windows"] = pd.to_numeric(df.loc[out.index, duration_col], errors="coerce").values
    else:
        out["duration_windows"] = np.nan
    out["duration_windows"] = out["duration_windows"].fillna(out["episode_end_bucket_id"] - out["episode_start_bucket_id"] + 1).astype(int)

    parsed = out["scenario_label"].apply(parse_scenario_label).apply(pd.Series)
    for c in parsed.columns:
        if c not in out.columns:
            out[c] = parsed[c]
    out["window_minutes"] = out["window_minutes"].fillna(out["scenario_label"].map(lambda s: get_window_minutes(s, 5))).astype(int)
    out["h_windows"] = out["h_windows"].fillna(out["scenario_label"].map(lambda s: get_h_windows(s, 12))).astype(int)
    out["duration_minutes"] = out["duration_windows"] * out["window_minutes"]
    out["duration_bucket"] = out["duration_windows"].map(duration_bucket_from_windows)
    out["duration_bucket_order"] = out["duration_bucket"].map(duration_bucket_order)
    return out.sort_values(["scenario_label", "episode_start_bucket_id", "episode_end_bucket_id"]).reset_index(drop=True)


def standardize_states(df, cell_id):
    if df is None or df.empty:
        return pd.DataFrame()
    df = df.copy()
    cell_col = infer_column(df, ["cell_id", "cell", "borg_cell"], required=False, label="cell_id em estados")
    if cell_col:
        df = df[df[cell_col].astype(str).str.lower().str.replace("cell_", "", regex=False).eq(str(cell_id).lower())].copy()
    if df.empty:
        return pd.DataFrame()
    scenario_col = infer_column(df, ["scenario_label", "scenario", "scenario_id", "config_label"], label="cenário em estados")
    bucket_col = infer_column(df, ["bucket_id", "bucket", "time_order", "pos"], label="bucket_id em estados")
    state_col = infer_column(df, ["state", "operational_state", "estado", "S"], required=False, label="estado operacional")
    split_col = infer_column(df, ["split"], required=False, label="split")
    out = pd.DataFrame({
        "cell_id": str(cell_id),
        "scenario_label": df[scenario_col].astype(str).values,
        "bucket_id": pd.to_numeric(df[bucket_col], errors="coerce").values,
        "state_nb10": df[state_col].astype(str).values if state_col else "unknown",
        "split": df[split_col].astype(str).values if split_col else "unknown",
    })
    out = out.dropna(subset=["bucket_id"])
    out["bucket_id"] = out["bucket_id"].astype(int)
    return out.sort_values(["scenario_label", "bucket_id"]).reset_index(drop=True)


# ============================================================
# 6. Métricas por tau, custo e antecipabilidade
# ============================================================


def confusion_counts(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    return tn, fp, fn, tp


def metrics_at_tau(scores_df, tau):
    y_true = scores_df["y_true"].astype(int).to_numpy()
    score = scores_df["score"].astype(float).to_numpy()
    y_pred = (score >= float(tau)).astype(int)
    tn, fp, fn, tp = confusion_counts(y_true, y_pred)
    n = int(len(y_true))
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    f1 = safe_div(2 * precision * recall, precision + recall) if not (pd.isna(precision) or pd.isna(recall)) else np.nan
    fpr = safe_div(fp, fp + tn)
    fnr = safe_div(fn, fn + tp)
    window_minutes = int(scores_df["window_minutes"].dropna().iloc[0]) if "window_minutes" in scores_df.columns and scores_df["window_minutes"].notna().any() else 5
    total_days = (n * window_minutes) / (60 * 24)
    return {
        "tau": float(tau),
        "n": n,
        "positive_rate": safe_div(y_true.sum(), n),
        "alert_rate": safe_div(y_pred.sum(), n),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "accuracy": safe_div(tp + tn, n),
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "fpr": fpr,
        "fnr": fnr,
        "false_alerts": int(fp),
        "false_alerts_per_day": safe_div(fp, total_days),
        "window_minutes": int(window_minutes),
        "total_scored_days": float(total_days),
    }


def compute_threshold_and_cost(scores_df):
    threshold_rows = []
    for (cell_id, scenario_label, score_source), s in scores_df.groupby(["cell_id", "scenario_label", "score_source"], dropna=False):
        if s.empty:
            continue
        meta = parse_scenario_label(scenario_label)
        for tau in TAU_GRID:
            row = metrics_at_tau(s, tau)
            row.update({
                "cell_id": cell_id,
                "scenario_label": scenario_label,
                "score_source": score_source,
                "score_label": f"{scenario_label} | {score_source}",
                "score_source_file": s["score_source_file"].iloc[0],
                "model": s["model"].iloc[0],
                "feature_set": s["feature_set"].iloc[0],
                **meta,
            })
            threshold_rows.append(row)
    df_threshold = pd.DataFrame(threshold_rows)

    cost_rows = []
    for _, row in df_threshold.iterrows():
        for cr in CONFIG["cost_ratios"]:
            cost = cr["c_fp"] * row["fpr"] + cr["c_fn"] * row["fnr"]
            cost_norm = safe_div(cost, cr["c_fp"] + cr["c_fn"])
            cost_rows.append({
                "cell_id": row["cell_id"],
                "scenario_label": row["scenario_label"],
                "score_source": row["score_source"],
                "score_label": row["score_label"],
                "tau": row["tau"],
                "cost_label": cr["cost_label"],
                "c_fp": cr["c_fp"],
                "c_fn": cr["c_fn"],
                "cost": cost,
                "cost_normalized": cost_norm,
                "fpr": row["fpr"],
                "fnr": row["fnr"],
                "precision": row["precision"],
                "recall": row["recall"],
                "f1": row["f1"],
                "false_alerts_per_day": row["false_alerts_per_day"],
                "model": row["model"],
                "feature_set": row["feature_set"],
            })
    df_cost = pd.DataFrame(cost_rows)

    optimal_rows = []
    if not df_cost.empty:
        for keys, grp in df_cost.groupby(["cell_id", "scenario_label", "score_source", "cost_label"], dropna=False):
            opt = grp.sort_values(["cost", "fnr", "fpr", "tau"], ascending=[True, True, True, True]).iloc[0].to_dict()
            opt["optimality_rule"] = "minimize_cFP_FPR_plus_cFN_FNR_then_low_FNR_low_FPR_low_tau"
            optimal_rows.append(opt)
    df_opt = pd.DataFrame(optimal_rows)

    false_cols = [
        "cell_id", "scenario_label", "score_source", "score_label", "tau",
        "false_alerts", "false_alerts_per_day", "alert_rate", "positive_rate",
        "fp", "tn", "fn", "tp", "model", "feature_set",
    ]
    df_false = df_threshold[[c for c in false_cols if c in df_threshold.columns]].copy() if not df_threshold.empty else pd.DataFrame()
    return df_threshold, df_cost, df_opt, df_false


def compute_episode_detail_for_tau(episodes_s, scores_s, tau):
    scores_s = scores_s.sort_values("bucket_id").copy()
    score_lookup = dict(zip(scores_s["bucket_id"].astype(int), scores_s["score"].astype(float)))
    score_bucket_set = set(score_lookup.keys())
    source = scores_s["score_source"].iloc[0] if "score_source" in scores_s.columns and not scores_s.empty else "unknown"
    rows = []
    for _, ep in episodes_s.iterrows():
        scenario_label = ep["scenario_label"]
        start = int(ep["episode_start_bucket_id"])
        end = int(ep["episode_end_bucket_id"])
        h_windows = int(ep.get("h_windows", get_h_windows(scenario_label, 12)))
        window_minutes = int(ep.get("window_minutes", get_window_minutes(scenario_label, 5)))

        previous_window = list(range(start - h_windows, start))
        available_previous = [b for b in previous_window if b in score_bucket_set]
        is_evaluable_full = len(available_previous) == h_windows
        is_evaluable_any = len(available_previous) > 0

        alert_buckets = [b for b in available_previous if score_lookup.get(b, -np.inf) >= float(tau)]
        if alert_buckets:
            first_alert_bucket = int(min(alert_buckets))
            lead_time_windows = int(start - first_alert_bucket)
            lead_time_minutes = int(lead_time_windows * window_minutes)
            anticipated_any = 1
            anticipated_full = int(is_evaluable_full)
        else:
            first_alert_bucket = np.nan
            lead_time_windows = np.nan
            lead_time_minutes = np.nan
            anticipated_any = 0
            anticipated_full = 0

        during_window = list(range(start, end + 1))
        available_during = [b for b in during_window if b in score_bucket_set]
        late_alert_buckets = [b for b in available_during if score_lookup.get(b, -np.inf) >= float(tau)]

        rows.append({
            "cell_id": ep.get("cell_id", ""),
            "scenario_label": scenario_label,
            "score_source": source,
            "score_label": f"{scenario_label} | {source}",
            "tau": float(tau),
            "episode_start_bucket_id": start,
            "episode_end_bucket_id": end,
            "duration_windows": int(ep["duration_windows"]),
            "duration_minutes": int(ep["duration_minutes"]),
            "duration_bucket": ep["duration_bucket"],
            "duration_bucket_order": int(ep["duration_bucket_order"]),
            "window_minutes": window_minutes,
            "h_windows": h_windows,
            "available_previous_windows": int(len(available_previous)),
            "evaluable_full_horizon": int(is_evaluable_full),
            "evaluable_any_score": int(is_evaluable_any),
            "anticipated_full_horizon": int(anticipated_full),
            "anticipated_any_score": int(anticipated_any),
            "first_alert_bucket_id": first_alert_bucket,
            "lead_time_windows": lead_time_windows,
            "lead_time_minutes": lead_time_minutes,
            "detected_during": int(len(late_alert_buckets) > 0),
            "available_during_windows": int(len(available_during)),
        })
    return pd.DataFrame(rows)


def compute_episode_anticipability(episodes_df, scores_df, optimal_tau_df):
    episode_tau_summary_rows = []
    episode_detail_selected = []
    group_cols = ["cell_id", "scenario_label", "score_source"]
    for (cell_id, scenario_label, score_source), scores_s in scores_df.groupby(group_cols, dropna=False):
        eps_s = episodes_df[
            (episodes_df["cell_id"].astype(str) == str(cell_id))
            & (episodes_df["scenario_label"].astype(str) == str(scenario_label))
        ].copy()
        if eps_s.empty or scores_s.empty:
            continue
        for tau in TAU_GRID:
            detail = compute_episode_detail_for_tau(eps_s, scores_s, tau)
            total_episodes = len(detail)
            evaluable_full = int(detail["evaluable_full_horizon"].sum()) if not detail.empty else 0
            anticipated_full = int(detail["anticipated_full_horizon"].sum()) if not detail.empty else 0
            evaluable_any = int(detail["evaluable_any_score"].sum()) if not detail.empty else 0
            anticipated_any = int(detail["anticipated_any_score"].sum()) if not detail.empty else 0
            lead_full = detail[(detail["anticipated_full_horizon"] == 1) & detail["lead_time_minutes"].notna()] if not detail.empty else pd.DataFrame()
            episode_tau_summary_rows.append({
                "cell_id": cell_id,
                "scenario_label": scenario_label,
                "score_source": score_source,
                "score_label": f"{scenario_label} | {score_source}",
                "tau": float(tau),
                "n_total_episodes": int(total_episodes),
                "n_evaluable_full_horizon": evaluable_full,
                "n_anticipated_full_horizon": anticipated_full,
                "episode_anticipation_rate_full_horizon": safe_div(anticipated_full, evaluable_full),
                "n_evaluable_any_score": evaluable_any,
                "n_anticipated_any_score": anticipated_any,
                "episode_anticipation_rate_any_score": safe_div(anticipated_any, evaluable_any),
                "lead_time_minutes_mean": float(lead_full["lead_time_minutes"].mean()) if not lead_full.empty else np.nan,
                "lead_time_minutes_median": float(lead_full["lead_time_minutes"].median()) if not lead_full.empty else np.nan,
                "lead_time_windows_mean": float(lead_full["lead_time_windows"].mean()) if not lead_full.empty else np.nan,
                "lead_time_windows_median": float(lead_full["lead_time_windows"].median()) if not lead_full.empty else np.nan,
            })

        opt_taus = optimal_tau_df[
            (optimal_tau_df["cell_id"].astype(str) == str(cell_id))
            & (optimal_tau_df["scenario_label"].astype(str) == str(scenario_label))
            & (optimal_tau_df["score_source"].astype(str) == str(score_source))
        ]["tau"].dropna().unique().tolist() if not optimal_tau_df.empty else []
        selected_taus = sorted(set([float(CONFIG["classification_reference_tau"])] + [float(t) for t in opt_taus]))
        for tau in selected_taus:
            detail = compute_episode_detail_for_tau(eps_s, scores_s, tau)
            cost_labels_for_tau = optimal_tau_df[
                (optimal_tau_df["cell_id"].astype(str) == str(cell_id))
                & (optimal_tau_df["scenario_label"].astype(str) == str(scenario_label))
                & (optimal_tau_df["score_source"].astype(str) == str(score_source))
                & (np.isclose(optimal_tau_df["tau"].astype(float), float(tau)))
            ]["cost_label"].dropna().unique().tolist() if not optimal_tau_df.empty else []
            detail["tau_role"] = "reference_0.50" if np.isclose(tau, CONFIG["classification_reference_tau"]) else "optimal_cost_tau"
            detail["cost_labels_at_tau"] = "|".join(cost_labels_for_tau) if cost_labels_for_tau else ""
            episode_detail_selected.append(detail)

    df_episode_tau_summary = pd.DataFrame(episode_tau_summary_rows)
    df_episode_detail = pd.concat(episode_detail_selected, ignore_index=True) if episode_detail_selected else pd.DataFrame()
    return df_episode_tau_summary, df_episode_detail


def compute_duration_summary(df_episode_detail):
    rows = []
    if df_episode_detail is None or df_episode_detail.empty:
        return pd.DataFrame()
    group_cols = ["cell_id", "scenario_label", "score_source", "tau", "tau_role", "duration_bucket"]
    for keys, grp in df_episode_detail.groupby(group_cols, dropna=False):
        cell_id, scenario_label, score_source, tau, tau_role, duration_bucket = keys
        grp = grp.copy()
        n_total = int(len(grp))
        n_eval = int(grp["evaluable_full_horizon"].sum())
        n_ant = int(grp["anticipated_full_horizon"].sum())
        ant = grp[(grp["anticipated_full_horizon"] == 1) & grp["lead_time_minutes"].notna()]
        rows.append({
            "cell_id": cell_id,
            "scenario_label": scenario_label,
            "score_source": score_source,
            "score_label": f"{scenario_label} | {score_source}",
            "tau": float(tau),
            "tau_role": tau_role,
            "duration_bucket": duration_bucket,
            "duration_bucket_order": int(grp["duration_bucket_order"].iloc[0]) if "duration_bucket_order" in grp.columns else duration_bucket_order(duration_bucket),
            "n_total_episodes": n_total,
            "n_evaluable_full_horizon": n_eval,
            "n_anticipated_full_horizon": n_ant,
            "episode_recall_by_duration": safe_div(n_ant, n_eval),
            "lead_time_minutes_mean": float(ant["lead_time_minutes"].mean()) if not ant.empty else np.nan,
            "lead_time_minutes_median": float(ant["lead_time_minutes"].median()) if not ant.empty else np.nan,
            "lead_time_windows_mean": float(ant["lead_time_windows"].mean()) if not ant.empty else np.nan,
            "lead_time_windows_median": float(ant["lead_time_windows"].median()) if not ant.empty else np.nan,
            "low_count_flag": int(n_eval < CONFIG["low_count_threshold_duration_bin"]),
            "interpretation_flag": "descritivo_baixa_contagem" if n_eval < CONFIG["low_count_threshold_duration_bin"] else "avaliavel",
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["cell_id", "scenario_label", "score_source", "tau", "duration_bucket_order"])
    return out


def compute_scenario_summary(scores_df, episodes_df, threshold_df, episode_tau_df, optimal_tau_df, primary_source):
    rows = []
    group_cols = ["cell_id", "scenario_label", "score_source"]
    for (cell_id, scenario_label, score_source), scores_s in scores_df.groupby(group_cols, dropna=False):
        eps_s = episodes_df[(episodes_df["cell_id"].astype(str) == str(cell_id)) & (episodes_df["scenario_label"].astype(str) == str(scenario_label))].copy()
        threshold_ref = threshold_df[
            (threshold_df["cell_id"].astype(str) == str(cell_id))
            & (threshold_df["scenario_label"].astype(str) == str(scenario_label))
            & (threshold_df["score_source"].astype(str) == str(score_source))
            & (np.isclose(threshold_df["tau"].astype(float), CONFIG["classification_reference_tau"]))
        ]
        episode_ref = episode_tau_df[
            (episode_tau_df["cell_id"].astype(str) == str(cell_id))
            & (episode_tau_df["scenario_label"].astype(str) == str(scenario_label))
            & (episode_tau_df["score_source"].astype(str) == str(score_source))
            & (np.isclose(episode_tau_df["tau"].astype(float), CONFIG["classification_reference_tau"]))
        ] if not episode_tau_df.empty else pd.DataFrame()
        opt_s = optimal_tau_df[
            (optimal_tau_df["cell_id"].astype(str) == str(cell_id))
            & (optimal_tau_df["scenario_label"].astype(str) == str(scenario_label))
            & (optimal_tau_df["score_source"].astype(str) == str(score_source))
        ].copy() if not optimal_tau_df.empty else pd.DataFrame()
        opt_dict = {}
        for _, r in opt_s.iterrows():
            opt_dict[f"tau_opt_{r['cost_label']}"] = float(r["tau"])
            opt_dict[f"cost_opt_{r['cost_label']}"] = float(r["cost"])
            opt_dict[f"recall_at_opt_{r['cost_label']}"] = float(r["recall"]) if pd.notna(r["recall"]) else np.nan
            opt_dict[f"fpr_at_opt_{r['cost_label']}"] = float(r["fpr"]) if pd.notna(r["fpr"]) else np.nan
            opt_dict[f"fnr_at_opt_{r['cost_label']}"] = float(r["fnr"]) if pd.notna(r["fnr"]) else np.nan

        row = {
            "cell_id": cell_id,
            "scenario_label": scenario_label,
            "score_source": score_source,
            "score_label": f"{scenario_label} | {score_source}",
            "is_primary_score_source": int(str(score_source) == str(primary_source)),
            "score_source_file": scores_s["score_source_file"].iloc[0] if not scores_s.empty else "",
            "model": scores_s["model"].iloc[0] if not scores_s.empty else "",
            "feature_set": scores_s["feature_set"].iloc[0] if not scores_s.empty else "",
            "n_scored_rows": int(len(scores_s)),
            "n_total_episodes": int(len(eps_s)),
            "duration_one_window_pct": safe_div((eps_s["duration_windows"] == 1).sum(), len(eps_s)) if not eps_s.empty else np.nan,
        }
        if not threshold_ref.empty:
            tr = threshold_ref.iloc[0]
            row.update({
                "tau_reference": float(CONFIG["classification_reference_tau"]),
                "precision_tau_reference": tr["precision"],
                "recall_tau_reference": tr["recall"],
                "f1_tau_reference": tr["f1"],
                "fpr_tau_reference": tr["fpr"],
                "fnr_tau_reference": tr["fnr"],
                "false_alerts_per_day_tau_reference": tr["false_alerts_per_day"],
                "positive_rate_tau_reference": tr["positive_rate"],
                "alert_rate_tau_reference": tr["alert_rate"],
            })
        if not episode_ref.empty:
            er = episode_ref.iloc[0]
            row.update({
                "n_evaluable_episodes_tau_reference": int(er["n_evaluable_full_horizon"]),
                "n_anticipated_episodes_tau_reference": int(er["n_anticipated_full_horizon"]),
                "episode_anticipation_rate_tau_reference": er["episode_anticipation_rate_full_horizon"],
                "lead_time_minutes_mean_tau_reference": er["lead_time_minutes_mean"],
                "lead_time_minutes_median_tau_reference": er["lead_time_minutes_median"],
            })
        row.update(opt_dict)
        rows.append(row)
    return pd.DataFrame(rows)


# ============================================================
# 7. Figuras por célula e aggregate
# ============================================================


def save_current_figure(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()
    return path


def make_cell_figures(cell_id, cell_fig_dir, main_scenario, primary_source, scores_df, episodes_df, threshold_df, cost_df, opt_df, duration_df):
    fig_paths = {}
    if not CONFIG.get("save_figures", True):
        return fig_paths
    cell_fig_dir = ensure_dir(cell_fig_dir)
    main_score = f"{main_scenario} | {primary_source}"

    try:
        main_cost = cost_df[(cost_df["score_label"] == main_score)].copy()
        if main_cost.empty and not cost_df.empty:
            main_cost = cost_df[cost_df["score_source"].astype(str).eq(str(primary_source))].copy()
        if not main_cost.empty:
            plt.figure(figsize=(10, 6))
            for cost_label, grp in main_cost.groupby("cost_label"):
                grp = grp.sort_values("tau")
                plt.plot(grp["tau"], grp["cost"], label=cost_label)
            plt.xlabel("tau")
            plt.ylabel("C(tau) = cFP × FPR + cFN × FNR")
            plt.title(f"NB14_FULL — Curvas de custo — cell_{cell_id}")
            plt.legend()
            fig_paths["cost_curve"] = str(save_current_figure(cell_fig_dir / f"14_FULL_fig01_cost_curves_cell_{cell_id}.png"))
    except Exception as exc:
        log(f"[AVISO] Figura custo cell_{cell_id}: {exc}")

    try:
        main_metrics = threshold_df[(threshold_df["score_label"] == main_score)].copy()
        if main_metrics.empty and not threshold_df.empty:
            main_metrics = threshold_df[threshold_df["score_source"].astype(str).eq(str(primary_source))].copy()
        if not main_metrics.empty:
            main_metrics = main_metrics.sort_values("tau")
            plt.figure(figsize=(10, 6))
            for metric in ["precision", "recall", "f1"]:
                plt.plot(main_metrics["tau"], main_metrics[metric], label=metric)
            plt.xlabel("tau")
            plt.ylabel("Métrica")
            plt.title(f"NB14_FULL — Trade-off por limiar — cell_{cell_id}")
            plt.legend()
            fig_paths["metrics_tradeoff"] = str(save_current_figure(cell_fig_dir / f"14_FULL_fig02_tau_metrics_tradeoff_cell_{cell_id}.png"))
    except Exception as exc:
        log(f"[AVISO] Figura métricas cell_{cell_id}: {exc}")

    try:
        fig_df = duration_df[(duration_df["score_label"] == main_score) & (duration_df["tau_role"] == "reference_0.50")].copy()
        if fig_df.empty:
            fig_df = duration_df[(duration_df["score_source"].astype(str).eq(str(primary_source))) & (duration_df["tau_role"] == "reference_0.50")].copy()
        if not fig_df.empty:
            fig_df = fig_df.sort_values("duration_bucket_order")
            plt.figure(figsize=(8, 5))
            plt.bar(fig_df["duration_bucket"], fig_df["episode_recall_by_duration"])
            plt.xlabel("Faixa de duração")
            plt.ylabel("Taxa de antecipação")
            plt.title(f"NB14_FULL — Antecipabilidade por duração — cell_{cell_id}")
            plt.xticks(rotation=30)
            fig_paths["duration_anticipability"] = str(save_current_figure(cell_fig_dir / f"14_FULL_fig03_duration_anticipability_cell_{cell_id}.png"))
    except Exception as exc:
        log(f"[AVISO] Figura duração cell_{cell_id}: {exc}")

    try:
        fig_df = duration_df[(duration_df["score_label"] == main_score) & (duration_df["tau_role"] == "reference_0.50")].copy()
        if fig_df.empty:
            fig_df = duration_df[(duration_df["score_source"].astype(str).eq(str(primary_source))) & (duration_df["tau_role"] == "reference_0.50")].copy()
        if not fig_df.empty:
            fig_df = fig_df.sort_values("duration_bucket_order")
            plt.figure(figsize=(8, 5))
            plt.bar(fig_df["duration_bucket"], fig_df["lead_time_minutes_median"])
            plt.xlabel("Faixa de duração")
            plt.ylabel("Lead time mediano (min)")
            plt.title(f"NB14_FULL — Lead time por duração — cell_{cell_id}")
            plt.xticks(rotation=30)
            fig_paths["lead_time_by_duration"] = str(save_current_figure(cell_fig_dir / f"14_FULL_fig04_lead_time_by_duration_cell_{cell_id}.png"))
    except Exception as exc:
        log(f"[AVISO] Figura lead time cell_{cell_id}: {exc}")

    try:
        scores_plot = scores_df[(scores_df["scenario_label"] == main_scenario) & (scores_df["score_source"] == primary_source)].copy()
        eps_plot = episodes_df[episodes_df["scenario_label"] == main_scenario].copy()
        if scores_plot.empty:
            scores_plot = scores_df[scores_df["score_source"] == primary_source].copy()
        if not scores_plot.empty:
            score_min = int(scores_plot["bucket_id"].min())
            score_max = int(scores_plot["bucket_id"].max())
            h_windows_plot = int(scores_plot["h_windows"].dropna().iloc[0]) if scores_plot["h_windows"].notna().any() else 12
            eps_evaluable = eps_plot[(eps_plot["episode_start_bucket_id"] - h_windows_plot >= score_min) & (eps_plot["episode_start_bucket_id"] <= score_max)].copy()
            if not eps_evaluable.empty:
                chosen_ep = eps_evaluable.iloc[0]
                start_plot = max(score_min, int(chosen_ep["episode_start_bucket_id"]) - CONFIG["timeline_context_h_multiplier"] * h_windows_plot)
                end_plot = min(score_max, int(chosen_ep["episode_end_bucket_id"]) + CONFIG["timeline_context_h_multiplier"] * h_windows_plot)
            else:
                start_plot = score_min
                end_plot = min(score_max, score_min + 300)
            seg = scores_plot[(scores_plot["bucket_id"] >= start_plot) & (scores_plot["bucket_id"] <= end_plot)].copy()
            opt_main = opt_df[(opt_df["scenario_label"] == main_scenario) & (opt_df["score_source"] == primary_source) & (opt_df["cost_label"] == "cFP1_cFN10")]
            tau_opt_10 = float(opt_main["tau"].iloc[0]) if not opt_main.empty else CONFIG["classification_reference_tau"]
            if not seg.empty:
                plt.figure(figsize=(12, 6))
                plt.plot(seg["bucket_id"], seg["score"], label="escore")
                plt.axhline(CONFIG["classification_reference_tau"], linestyle="--", label="tau=0,50")
                plt.axhline(tau_opt_10, linestyle=":", label="tau ótimo 1:10")
                eps_seg = eps_plot[(eps_plot["episode_end_bucket_id"] >= start_plot) & (eps_plot["episode_start_bucket_id"] <= end_plot)].copy()
                for _, ep in eps_seg.iterrows():
                    plt.axvspan(int(ep["episode_start_bucket_id"]), int(ep["episode_end_bucket_id"]), alpha=0.15)
                plt.xlabel("bucket_id")
                plt.ylabel("Escore")
                plt.title(f"NB14_FULL — Escores e episódios — cell_{cell_id}")
                plt.legend()
                fig_paths["timeline_scores"] = str(save_current_figure(cell_fig_dir / f"14_FULL_fig05_timeline_scores_cell_{cell_id}.png"))
    except Exception as exc:
        log(f"[AVISO] Figura timeline cell_{cell_id}: {exc}")
    return fig_paths


def make_aggregate_figures(summary_by_cell, opt_all, agg_fig_dir):
    fig_paths = {}
    if not CONFIG.get("save_figures", True):
        return fig_paths
    agg_fig_dir = ensure_dir(agg_fig_dir)
    primary = summary_by_cell[summary_by_cell.get("is_primary_score_source", 0).eq(1)].copy() if "is_primary_score_source" in summary_by_cell.columns else summary_by_cell.copy()
    if primary.empty:
        primary = summary_by_cell.copy()

    try:
        if not primary.empty and "episode_anticipation_rate_tau_reference" in primary.columns:
            d = primary.sort_values("cell_id")
            plt.figure(figsize=(10, 5))
            plt.bar(d["cell_id"].astype(str), d["episode_anticipation_rate_tau_reference"])
            plt.xlabel("cell_id")
            plt.ylabel("Taxa de antecipação de episódios (tau=0,50)")
            plt.title("NB14_FULL — Antecipabilidade por célula")
            fig_paths["episode_anticipability_by_cell"] = str(save_current_figure(agg_fig_dir / "14_FULL_figA_episode_anticipability_by_cell.png"))
    except Exception as exc:
        log(f"[AVISO] Figura aggregate antecipabilidade: {exc}")

    try:
        if not primary.empty and "false_alerts_per_day_tau_reference" in primary.columns:
            d = primary.sort_values("cell_id")
            plt.figure(figsize=(10, 5))
            plt.bar(d["cell_id"].astype(str), d["false_alerts_per_day_tau_reference"])
            plt.xlabel("cell_id")
            plt.ylabel("Falsos alertas por dia (tau=0,50)")
            plt.title("NB14_FULL — Falsos alertas por dia por célula")
            fig_paths["false_alerts_by_cell"] = str(save_current_figure(agg_fig_dir / "14_FULL_figB_false_alerts_by_cell.png"))
    except Exception as exc:
        log(f"[AVISO] Figura aggregate falsos alertas: {exc}")

    try:
        if not opt_all.empty:
            d = opt_all[(opt_all["cost_label"] == "cFP1_cFN10")].copy()
            if "is_primary_score_source" in summary_by_cell.columns:
                prim_keys = primary[["cell_id", "scenario_label", "score_source"]].drop_duplicates()
                d = d.merge(prim_keys, on=["cell_id", "scenario_label", "score_source"], how="inner")
            if not d.empty:
                d = d.sort_values("cell_id")
                plt.figure(figsize=(10, 5))
                plt.bar(d["cell_id"].astype(str), d["tau"])
                plt.xlabel("cell_id")
                plt.ylabel("tau ótimo — custo 1:10")
                plt.title("NB14_FULL — tau ótimo por célula")
                fig_paths["tau_opt_by_cell"] = str(save_current_figure(agg_fig_dir / "14_FULL_figC_tau_opt_by_cell.png"))
    except Exception as exc:
        log(f"[AVISO] Figura aggregate tau ótimo: {exc}")
    return fig_paths


# ============================================================
# 8. Processamento por célula
# ============================================================


def resolve_and_load_score_frames(cell_id, primary_family):
    frames = []
    diagnostics = []

    families_to_try = []
    # Fonte primária primeiro.
    families_to_try.append((primary_family, True))
    # NB11 como fallback obrigatório se a fonte primária LSTM falhar.
    if primary_family != "nb11":
        families_to_try.append(("nb11", True))
    # Sensibilidades opcionais.
    if CONFIG.get("include_nb12_sensitivity_scores", True):
        families_to_try.append(("nb12", False))
    if CONFIG.get("include_lstm_optional_sensitivity_scores", True):
        for fam in ["nb13", "nb13a"]:
            if fam != primary_family:
                families_to_try.append((fam, False))

    seen = set()
    for fam, is_required_candidate in families_to_try:
        if fam in seen:
            continue
        seen.add(fam)
        path = resolve_score_file_for_family(cell_id, fam)
        if path is None:
            diagnostics.append({"family": fam, "status": "missing", "path": None})
            continue
        try:
            raw = pd.read_parquet(path)
            std = standardize_score_frame(raw, cell_id=cell_id, source_name=f"{fam.upper()}_FULL", source_path=path)
            frames.append(std)
            diagnostics.append({"family": fam, "status": "loaded", "path": str(path), "rows": int(len(std))})
        except Exception as exc:
            diagnostics.append({"family": fam, "status": "error", "path": str(path), "error": repr(exc)})
            if is_required_candidate and fam == primary_family and primary_family != "nb11":
                log(f"[AVISO] Fonte primária {fam} falhou para cell_{cell_id}; tentando NB11_FULL como fallback. Erro: {exc}")
            elif is_required_candidate and fam == "nb11":
                raise

    if not frames:
        raise FileNotFoundError(f"Nenhum score frame carregado para cell_{cell_id}. Diagnóstico: {diagnostics}")

    scores = pd.concat(frames, ignore_index=True)
    # Define se a fonte primária real foi carregada; se não, usa NB11_FULL como primária.
    primary_source = f"{primary_family.upper()}_FULL"
    if primary_source not in set(scores["score_source"].astype(str)):
        if "NB11_FULL" in set(scores["score_source"].astype(str)):
            primary_source = "NB11_FULL"
        else:
            primary_source = scores["score_source"].iloc[0]
    return scores, primary_source, diagnostics


def process_cell(cell_id):
    log("-" * 100)
    log(f"Processando cell_{cell_id}")
    cell_dir = ensure_dir(STAGE14_DIR / f"cell_{cell_id}")
    cell_fig_dir = ensure_dir(cell_dir / "figures")

    nb11_ctx = select_nb11_context_for_cell(cell_id)
    primary_family, primary_reason, decision_payload = recommend_primary_score_family(cell_id)
    artifacts10 = resolve_stage10_artifacts(cell_id)

    if artifacts10["episodes"] is None:
        raise FileNotFoundError(f"Episódios NB10_FULL não encontrados para cell_{cell_id}. Diretório: {artifacts10['cell_dir']}")
    if artifacts10["states"] is None:
        raise FileNotFoundError(f"Estados NB10_FULL não encontrados para cell_{cell_id}. Diretório: {artifacts10['cell_dir']}")

    df_episodes = standardize_episodes(pd.read_parquet(artifacts10["episodes"]), cell_id)
    df_states = standardize_states(pd.read_parquet(artifacts10["states"]), cell_id)
    df_scores, primary_score_source, score_diagnostics = resolve_and_load_score_frames(cell_id, primary_family)

    # Cenários avaliados: principal do NB11 + P95 quando existir + qualquer cenário presente na fonte primária.
    main_scenario = str(nb11_ctx.get("scenario_label") or CONFIG["main_scenario_default"])
    p95_scenario = main_scenario.replace("TRAIN_M2S", "TRAIN_P95") if "TRAIN_M2S" in main_scenario else CONFIG["p95_scenario_default"]
    available = sorted(df_scores["scenario_label"].dropna().astype(str).unique().tolist())
    run_scenarios = []
    for s in [main_scenario, p95_scenario]:
        if s in available and s not in run_scenarios:
            run_scenarios.append(s)
    # Se o principal não está nos escores, usa o primeiro disponível como fallback explícito.
    if not run_scenarios and available:
        run_scenarios = [available[0]]
        main_scenario = available[0]
    if not run_scenarios:
        raise ValueError(f"Nenhum cenário disponível em escores para cell_{cell_id}")

    df_scores = df_scores[df_scores["scenario_label"].isin(run_scenarios)].copy()
    df_episodes = df_episodes[df_episodes["scenario_label"].isin(run_scenarios)].copy()
    df_states = df_states[df_states["scenario_label"].isin(run_scenarios)].copy() if not df_states.empty else df_states

    if df_episodes.empty:
        raise ValueError(f"Nenhum episódio NB10_FULL nos cenários {run_scenarios} para cell_{cell_id}")
    if df_scores.empty:
        raise ValueError(f"Nenhum escore nos cenários {run_scenarios} para cell_{cell_id}")

    df_threshold, df_cost, df_opt, df_false = compute_threshold_and_cost(df_scores)
    df_episode_tau, df_episode_detail = compute_episode_anticipability(df_episodes, df_scores, df_opt)
    df_duration = compute_duration_summary(df_episode_detail)
    df_scenario_summary = compute_scenario_summary(df_scores, df_episodes, df_threshold, df_episode_tau, df_opt, primary_score_source)

    # Saídas por célula.
    files = {
        "scores_selected": cell_dir / f"14_FULL_scores_selected_cell_{cell_id}.parquet",
        "threshold_metrics": cell_dir / f"14_FULL_threshold_metrics_by_tau_cell_{cell_id}.csv",
        "cost_curve": cell_dir / f"14_FULL_cost_curve_cell_{cell_id}.csv",
        "optimal_tau": cell_dir / f"14_FULL_optimal_tau_by_cost_cell_{cell_id}.csv",
        "false_alerts": cell_dir / f"14_FULL_false_alerts_by_tau_cell_{cell_id}.csv",
        "episode_anticipability_by_tau": cell_dir / f"14_FULL_episode_anticipability_by_tau_cell_{cell_id}.csv",
        "episode_anticipability": cell_dir / f"14_FULL_episode_anticipability_cell_{cell_id}.csv",
        "episode_duration_summary": cell_dir / f"14_FULL_episode_duration_summary_cell_{cell_id}.csv",
        "scenario_summary": cell_dir / f"14_FULL_scenario_summary_cell_{cell_id}.csv",
        "summary_json": cell_dir / f"14_FULL_decision_analysis_summary_cell_{cell_id}.json",
        "conclusion_notes": cell_dir / f"14_FULL_conclusion_notes_cell_{cell_id}.txt",
    }

    save_parquet(df_scores, files["scores_selected"])
    save_csv(df_threshold, files["threshold_metrics"])
    save_csv(df_cost, files["cost_curve"])
    save_csv(df_opt, files["optimal_tau"])
    save_csv(df_false, files["false_alerts"])
    save_csv(df_episode_tau, files["episode_anticipability_by_tau"])
    save_csv(df_episode_detail, files["episode_anticipability"])
    save_csv(df_duration, files["episode_duration_summary"])
    save_csv(df_scenario_summary, files["scenario_summary"])

    figures = make_cell_figures(
        cell_id=cell_id,
        cell_fig_dir=cell_fig_dir,
        main_scenario=main_scenario,
        primary_source=primary_score_source,
        scores_df=df_scores,
        episodes_df=df_episodes,
        threshold_df=df_threshold,
        cost_df=df_cost,
        opt_df=df_opt,
        duration_df=df_duration,
    )

    # Notas automáticas por célula.
    lines = []
    lines.append(f"NB14_FULL — Notas automáticas — cell_{cell_id}")
    lines.append(f"Execução: {RUN_TIMESTAMP}")
    lines.append(f"Cenários avaliados: {run_scenarios}")
    lines.append(f"Fonte primária de escores: {primary_score_source}")
    lines.append(f"Motivo da fonte primária: {primary_reason}")
    lines.append(f"Artefato episódios: {artifacts10['episodes']}")
    lines.append(f"Artefato estados: {artifacts10['states']}")
    lines.append("")
    if not df_scenario_summary.empty:
        primary_rows = df_scenario_summary[df_scenario_summary["is_primary_score_source"].eq(1)].copy()
        if primary_rows.empty:
            primary_rows = df_scenario_summary.copy()
        for _, r in primary_rows.iterrows():
            lines.append(f"- {r['score_label']}:")
            lines.append(f"  - linhas pontuadas: {r.get('n_scored_rows', np.nan)}")
            lines.append(f"  - episódios totais: {r.get('n_total_episodes', np.nan)}")
            lines.append(f"  - recall tau=0,50: {r.get('recall_tau_reference', np.nan)}")
            lines.append(f"  - F1 tau=0,50: {r.get('f1_tau_reference', np.nan)}")
            lines.append(f"  - falsos alertas/dia tau=0,50: {r.get('false_alerts_per_day_tau_reference', np.nan)}")
            lines.append(f"  - episódios avaliáveis tau=0,50: {r.get('n_evaluable_episodes_tau_reference', np.nan)}")
            lines.append(f"  - episódios antecipados tau=0,50: {r.get('n_anticipated_episodes_tau_reference', np.nan)}")
            lines.append(f"  - taxa de antecipação tau=0,50: {r.get('episode_anticipation_rate_tau_reference', np.nan)}")
            lines.append(f"  - lead time mediano tau=0,50: {r.get('lead_time_minutes_median_tau_reference', np.nan)} min")
    files["conclusion_notes"].write_text("\n".join(lines), encoding="utf-8")

    generated_artifacts = {name: str(path) for name, path in files.items()}
    summary = {
        "notebook": "NB14_FULL",
        "version": NB14_FULL_VERSION,
        "cell_id": cell_id,
        "run_timestamp": RUN_TIMESTAMP,
        "execution_scope": EXECUTION_SCOPE,
        "nb11_context": nb11_ctx,
        "main_scenario": main_scenario,
        "p95_scenario": p95_scenario,
        "run_scenarios": run_scenarios,
        "primary_score_family_requested": primary_family,
        "primary_score_source": primary_score_source,
        "primary_score_reason": primary_reason,
        "decision_payload": decision_payload,
        "score_diagnostics": score_diagnostics,
        "stage10_artifacts": {k: str(v) if v is not None else None for k, v in artifacts10.items()},
        "n_scores_rows": int(len(df_scores)),
        "n_episodes_rows": int(len(df_episodes)),
        "n_states_rows": int(len(df_states)) if df_states is not None else 0,
        "n_threshold_rows": int(len(df_threshold)),
        "n_cost_rows": int(len(df_cost)),
        "n_episode_detail_rows": int(len(df_episode_detail)),
        "generated_artifacts": generated_artifacts,
        "generated_figures": figures,
        "methodological_notes": [
            "Métricas calculadas por célula; não há concatenação temporal para avaliação.",
            "Tau é limiar sobre escore relativo de risco temporal.",
            "C(tau) segue cFP*FPR + cFN*FNR, como no NB14 canônico.",
            "Episódios e estados são herdados do NB10_FULL; este notebook não recalcula criticidade.",
            "Campo soberano do NB13a_FULL para integração: nb14_score_source_recommendation.",
        ],
    }
    save_json(summary, files["summary_json"])

    return {
        "cell_id": cell_id,
        "status": "OK",
        "summary": summary,
        "scores": df_scores,
        "threshold": df_threshold,
        "cost": df_cost,
        "optimal": df_opt,
        "false_alerts": df_false,
        "episode_tau": df_episode_tau,
        "episode_detail": df_episode_detail,
        "duration": df_duration,
        "scenario_summary": df_scenario_summary,
        "files": files,
        "figures": figures,
    }


# ============================================================
# 9. Execução principal por célula
# ============================================================

cell_results = []
execution_errors = []

for cell_id in ACTIVE_CELLS:
    try:
        res = process_cell(cell_id)
        cell_results.append(res)
    except Exception as exc:
        err = {
            "cell_id": cell_id,
            "status": "ERROR",
            "error": repr(exc),
            "traceback": traceback.format_exc(),
        }
        execution_errors.append(err)
        log(f"[ERRO] cell_{cell_id}: {exc}")
        if CONFIG.get("fail_fast", False):
            raise

if not cell_results:
    save_csv(pd.DataFrame(execution_errors), AGGREGATE_DIR / "14_FULL_execution_errors.csv")
    raise RuntimeError("NB14_FULL não concluiu nenhuma célula. Verifique 14_FULL_execution_errors.csv.")

# ============================================================
# 10. Consolidação aggregate
# ============================================================


def concat_result_frame(key):
    frames = [r[key] for r in cell_results if isinstance(r.get(key), pd.DataFrame) and not r[key].empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

scores_all = concat_result_frame("scores")
threshold_all = concat_result_frame("threshold")
cost_all = concat_result_frame("cost")
optimal_all = concat_result_frame("optimal")
false_all = concat_result_frame("false_alerts")
episode_tau_all = concat_result_frame("episode_tau")
episode_detail_all = concat_result_frame("episode_detail")
duration_all = concat_result_frame("duration")
scenario_summary_by_cell = concat_result_frame("scenario_summary")

# Artefatos aggregate.
aggregate_files = {
    "scores_selected_allcells": AGGREGATE_DIR / "14_FULL_scores_selected_allcells.parquet",
    "threshold_metrics_allcells": AGGREGATE_DIR / "14_FULL_threshold_metrics_by_tau_allcells.csv",
    "cost_curve_allcells": AGGREGATE_DIR / "14_FULL_cost_curve_allcells.csv",
    "optimal_tau_allcells": AGGREGATE_DIR / "14_FULL_optimal_tau_by_cost_allcells.csv",
    "false_alerts_allcells": AGGREGATE_DIR / "14_FULL_false_alerts_by_tau_allcells.csv",
    "episode_anticipability_by_tau_allcells": AGGREGATE_DIR / "14_FULL_episode_anticipability_by_tau_allcells.csv",
    "episode_anticipability_allcells": AGGREGATE_DIR / "14_FULL_episode_anticipability_allcells.csv",
    "episode_duration_summary_allcells": AGGREGATE_DIR / "14_FULL_episode_duration_summary_allcells.csv",
    "scenario_summary_by_cell": AGGREGATE_DIR / "14_FULL_scenario_summary_by_cell.csv",
    "cost_summary_by_cell": AGGREGATE_DIR / "14_FULL_cost_summary_by_cell.csv",
    "execution_errors": AGGREGATE_DIR / "14_FULL_execution_errors.csv",
    "delta_vs_canonical": AGGREGATE_DIR / "14_FULL_delta_vs_canonical.csv",
    "pilot_validation_summary": AGGREGATE_DIR / "14_FULL_pilot_validation_summary.csv",
    "summary_json": AGGREGATE_DIR / "14_FULL_nb14_summary.json",
    "conclusion_notes": AGGREGATE_DIR / "14_FULL_conclusion_notes.txt",
    "artifact_manifest": AGGREGATE_DIR / "14_FULL_artifact_manifest_sha256.csv",
}

if not scores_all.empty:
    save_parquet(scores_all, aggregate_files["scores_selected_allcells"])
else:
    save_csv(pd.DataFrame([{"note": "sem scores allcells"}]), AGGREGATE_DIR / "14_FULL_scores_selected_allcells_EMPTY.csv")
save_csv(threshold_all, aggregate_files["threshold_metrics_allcells"])
save_csv(cost_all, aggregate_files["cost_curve_allcells"])
save_csv(optimal_all, aggregate_files["optimal_tau_allcells"])
save_csv(false_all, aggregate_files["false_alerts_allcells"])
save_csv(episode_tau_all, aggregate_files["episode_anticipability_by_tau_allcells"])
save_csv(episode_detail_all, aggregate_files["episode_anticipability_allcells"])
save_csv(duration_all, aggregate_files["episode_duration_summary_allcells"])
save_csv(scenario_summary_by_cell, aggregate_files["scenario_summary_by_cell"])
save_csv(pd.DataFrame(execution_errors), aggregate_files["execution_errors"])

# Cost summary citável: linhas primárias por célula + tau ótimo 1:5, 1:10, 1:20.
if not scenario_summary_by_cell.empty:
    cost_summary_by_cell = scenario_summary_by_cell.copy()
    if "is_primary_score_source" in cost_summary_by_cell.columns:
        prim = cost_summary_by_cell[cost_summary_by_cell["is_primary_score_source"].eq(1)].copy()
        if not prim.empty:
            cost_summary_by_cell = prim
else:
    cost_summary_by_cell = pd.DataFrame()
save_csv(cost_summary_by_cell, aggregate_files["cost_summary_by_cell"])

# Delta vs canonical — documentação governada.
delta_vs_canonical = pd.DataFrame([
    {
        "notebook_full": "14_decision_analysis_FULL.ipynb",
        "notebook_canonical": "14_decision_analysis.ipynb",
        "change_type": "I/O_PATH",
        "description": "Troca de 04-reports canônico por 04-reports/99_FULL_downstream/14_FULL_decision_analysis.",
        "justified": True,
    },
    {
        "notebook_full": "14_decision_analysis_FULL.ipynb",
        "notebook_canonical": "14_decision_analysis.ipynb",
        "change_type": "CELL_LOOP",
        "description": "Execução parametrizada por cell_id a–h, com isolamento das métricas por célula.",
        "justified": True,
    },
    {
        "notebook_full": "14_decision_analysis_FULL.ipynb",
        "notebook_canonical": "14_decision_analysis.ipynb",
        "change_type": "AGGREGATION_BY_CELL",
        "description": "Consolidação final em aggregate sem recalcular métricas sobre série temporal concatenada.",
        "justified": True,
    },
    {
        "notebook_full": "14_decision_analysis_FULL.ipynb",
        "notebook_canonical": "14_decision_analysis.ipynb",
        "change_type": "OUTPUT_PREFIX",
        "description": "Todos os artefatos usam prefixo 14_FULL_ para impedir sobrescrita canônica.",
        "justified": True,
    },
    {
        "notebook_full": "14_decision_analysis_FULL.ipynb",
        "notebook_canonical": "14_decision_analysis.ipynb",
        "change_type": "REPORTING_ONLY",
        "description": "Inclusão de decisão governada por nb14_score_source_recommendation do NB13a_FULL.",
        "justified": True,
    },
])
save_csv(delta_vs_canonical, aggregate_files["delta_vs_canonical"])

# Gates de validação.
pilot_rows = []
expected_cells = set(ACTIVE_CELLS)
completed_cells = {r["cell_id"] for r in cell_results}
error_cells = {e["cell_id"] for e in execution_errors}
pilot_rows.append({
    "check": "completed_requested_cells",
    "status": "OK" if expected_cells.issubset(completed_cells) else "WARN",
    "observed": sorted(completed_cells),
    "expected": sorted(expected_cells),
    "note": "Todas as células solicitadas devem concluir ou aparecer em execution_errors.",
})
pilot_rows.append({
    "check": "errors_recorded",
    "status": "OK" if len(error_cells) == 0 else "WARN",
    "observed": sorted(error_cells),
    "expected": [],
    "note": "WARN não invalida células concluídas, mas deve ser lido antes do NB15_FULL.",
})
pilot_rows.append({
    "check": "full_output_prefix",
    "status": "OK" if all(Path(p).name.startswith("14_FULL_") for p in aggregate_files.values()) else "FAIL",
    "observed": [Path(p).name for p in aggregate_files.values()],
    "expected": "prefixo 14_FULL_",
    "note": "Impede colisão com artefatos canônicos.",
})
pilot_rows.append({
    "check": "full_downstream_path",
    "status": "OK" if "99_FULL_downstream" in str(STAGE14_DIR) else "FAIL",
    "observed": str(STAGE14_DIR),
    "expected": "caminho contendo 99_FULL_downstream",
    "note": "Protege o pipeline canônico.",
})
# Backstop metodológico NB13/NB13a -> NB14: no achado FULL atual, nenhuma célula
# deve promover LSTM a fonte primária, pois o critério de robustez Delta/SE >= 2,0
# não foi atingido. Se esta trava falhar, revisar manualmente antes do NB15_FULL.
lstm_primary_cells = sorted({
    str(r.get("cell_id"))
    for r in cell_results
    if str(r.get("primary_score_family_requested", "")).lower() in {"nb13", "nb13a"}
    or str(r.get("primary_score_source", "")).upper().startswith(("NB13", "NB13A"))
})
pilot_rows.append({
    "check": "score_source_governance_conservative",
    "status": "OK" if not lstm_primary_cells else "FAIL",
    "observed": lstm_primary_cells,
    "expected": "nenhuma célula com LSTM como fonte primária (critério Delta/SE >= 2,0 não atingido em NB13_FULL)",
    "note": "Promoção de LSTM exige recomendação governada explícita; revisar manualmente se FAIL.",
})
if not scenario_summary_by_cell.empty:
    primary_count = int(scenario_summary_by_cell.get("is_primary_score_source", pd.Series(dtype=int)).sum()) if "is_primary_score_source" in scenario_summary_by_cell.columns else len(scenario_summary_by_cell)
else:
    primary_count = 0
pilot_rows.append({
    "check": "primary_summary_rows",
    "status": "OK" if primary_count >= len(completed_cells) else "WARN",
    "observed": primary_count,
    "expected": f">= {len(completed_cells)}",
    "note": "Esperado pelo menos um resumo primário por célula concluída.",
})
pilot_validation_summary = pd.DataFrame(pilot_rows)
save_csv(pilot_validation_summary, aggregate_files["pilot_validation_summary"])

# Figuras aggregate.
aggregate_figures = make_aggregate_figures(scenario_summary_by_cell, optimal_all, AGG_FIGURES_DIR)

# Notas automáticas aggregate.
lines = []
lines.append("NB14_FULL — Notas automáticas aggregate")
lines.append(f"Execução: {RUN_TIMESTAMP}")
lines.append(f"Células solicitadas: {ACTIVE_CELLS}")
lines.append(f"Células concluídas: {sorted(completed_cells)}")
lines.append(f"Células com erro: {sorted(error_cells)}")
lines.append("")
if not cost_summary_by_cell.empty:
    lines.append("Resumo primário por célula:")
    for _, r in cost_summary_by_cell.sort_values("cell_id").iterrows():
        lines.append(f"- cell_{r.get('cell_id')}: {r.get('score_label')}")
        lines.append(f"  - linhas pontuadas: {r.get('n_scored_rows', np.nan)}")
        lines.append(f"  - episódios totais: {r.get('n_total_episodes', np.nan)}")
        lines.append(f"  - recall tau=0,50: {r.get('recall_tau_reference', np.nan)}")
        lines.append(f"  - F1 tau=0,50: {r.get('f1_tau_reference', np.nan)}")
        lines.append(f"  - falsos alertas/dia tau=0,50: {r.get('false_alerts_per_day_tau_reference', np.nan)}")
        lines.append(f"  - taxa de antecipação de episódios tau=0,50: {r.get('episode_anticipation_rate_tau_reference', np.nan)}")
        lines.append(f"  - lead time mediano tau=0,50: {r.get('lead_time_minutes_median_tau_reference', np.nan)} min")
        for cost_label in ["cFP1_cFN5", "cFP1_cFN10", "cFP1_cFN20"]:
            if f"tau_opt_{cost_label}" in r:
                lines.append(f"  - tau ótimo {cost_label}: {r.get(f'tau_opt_{cost_label}', np.nan)}")
lines.append("")
lines.append("Observação: as métricas aggregate são uma consolidação tabular por célula. O notebook não recalcula desempenho sobre uma série temporal allcells concatenada.")
aggregate_files["conclusion_notes"].write_text("\n".join(lines), encoding="utf-8")

# Summary JSON aggregate antes do manifesto.
summary = {
    "notebook": "NB14_FULL",
    "version": NB14_FULL_VERSION,
    "title": "Análise de decisão por célula — antecipabilidade, lead time e C(tau)",
    "run_timestamp": RUN_TIMESTAMP,
    "execution_scope": EXECUTION_SCOPE,
    "active_cells": ACTIVE_CELLS,
    "completed_cells": sorted(completed_cells),
    "error_cells": sorted(error_cells),
    "n_cells_requested": len(ACTIVE_CELLS),
    "n_cells_completed": len(completed_cells),
    "n_execution_errors": len(execution_errors),
    "base_dir": str(BASE_DIR),
    "full_reports_dir": str(FULL_REPORTS_DIR),
    "stage14_dir": str(STAGE14_DIR),
    "aggregate_dir": str(AGGREGATE_DIR),
    "config": CONFIG,
    "n_scores_rows": int(len(scores_all)),
    "n_threshold_rows": int(len(threshold_all)),
    "n_cost_rows": int(len(cost_all)),
    "n_optimal_tau_rows": int(len(optimal_all)),
    "n_episode_detail_rows": int(len(episode_detail_all)),
    "n_scenario_summary_rows": int(len(scenario_summary_by_cell)),
    "aggregate_files": {k: str(v) for k, v in aggregate_files.items()},
    "aggregate_figures": aggregate_figures,
    "pilot_validation_status_counts": pilot_validation_summary["status"].value_counts().to_dict() if not pilot_validation_summary.empty else {},
    "cell_summaries": [r["summary"] for r in cell_results],
    "execution_errors": execution_errors,
    "methodological_notes": [
        "Ramo _FULL é experimento ampliado de robustez/escala, não substituto do canônico.",
        "Cada célula Borg é uma réplica independente.",
        "C(tau) calculado por célula a partir de FPR e FNR.",
        "Episódios e estados herdados do NB10_FULL.",
        "Fonte primária conservadora é NB11_FULL, salvo recomendação soberana do NB13a_FULL.",
        "Aggregate sumariza resultados por célula; não há nova série temporal concatenada.",
    ],
}
save_json(summary, aggregate_files["summary_json"])

manifest_df = build_artifact_manifest(STAGE14_DIR, aggregate_files["artifact_manifest"])
summary["artifact_manifest_rows"] = int(len(manifest_df))
summary["artifact_manifest_path"] = str(aggregate_files["artifact_manifest"])
save_json(summary, aggregate_files["summary_json"])

# ============================================================
# 11. Relatório de encerramento
# ============================================================

print("\n" + "=" * 110)
print("NB14_FULL concluído.")
print("=" * 110)
print(f"Células solicitadas: {ACTIVE_CELLS}")
print(f"Células concluídas: {sorted(completed_cells)}")
print(f"Células com erro: {sorted(error_cells)}")

print("\nGates de validação/governança:")
display(pilot_validation_summary)

print("\nResumo primário por célula:")
display(cost_summary_by_cell)

print("\nTaus ótimos por célula/cenário/fonte/custo:")
display(optimal_all.sort_values(["cell_id", "scenario_label", "score_source", "cost_label"]) if not optimal_all.empty else optimal_all)

print("\nArtefatos aggregate:")
display(pd.DataFrame([
    {"artifact": name, "path": str(path), "exists": Path(path).exists()}
    for name, path in aggregate_files.items()
]))

if execution_errors:
    print("\n[AVISO] Houve erros em uma ou mais células. Consulte 14_FULL_execution_errors.csv antes do NB15_FULL.")
    display(pd.DataFrame(execution_errors))
else:
    print("\nTodas as células processadas concluíram sem erro registrado.")

print(
    "\nLeitura preliminar: o NB14_FULL converteu escores por célula em análise de limiar tau, "
    "custo C(tau), falsos alertas por dia, antecipabilidade de episódios e lead time. "
    "A integração com NB13a_FULL respeita nb14_score_source_recommendation. "
    "O NB15_FULL deve consumir prioritariamente 14_FULL_scenario_summary_by_cell.csv, "
    "14_FULL_cost_summary_by_cell.csv, 14_FULL_optimal_tau_by_cost_allcells.csv e as figuras aggregate."
)


Mounted at /content/drive
[2026-06-27 00:31:43] ==============================================================================================================
[2026-06-27 00:31:43] NB14_FULL — Análise de decisão por célula
[2026-06-27 00:31:43] NB14_FULL_VERSION = FULL_DECISION_ANALYSIS_BY_CELL_GOVERNED_V2
[2026-06-27 00:31:43] RUN_TIMESTAMP = 2026-06-27 00:31:10
[2026-06-27 00:31:43] BASE_DIR = /content/drive/MyDrive/Mestrado
[2026-06-27 00:31:43] FULL_REPORTS_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream
[2026-06-27 00:31:43] STAGE10_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/10_FULL_threshold_diagnostics
[2026-06-27 00:31:43] STAGE11_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/11_FULL_model_baselines
[2026-06-27 00:31:43] STAGE12_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/12_FULL_sensitivity
[2026-06-27 00:31:43] STAGE13_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/

,check,status,observed,expected,note
0,completed_requested_cells,OK,"[a, b, c, d, e, f, g, h]","[a, b, c, d, e, f, g, h]",Todas as células solicitadas devem concluir ou...
1,errors_recorded,OK,[],[],"WARN não invalida células concluídas, mas deve..."
2,full_output_prefix,OK,"[14_FULL_scores_selected_allcells.parquet, 14_...",prefixo 14_FULL_,Impede colisão com artefatos canônicos.
3,full_downstream_path,OK,/content/drive/MyDrive/Mestrado/04-reports/99_...,caminho contendo 99_FULL_downstream,Protege o pipeline canônico.
4,score_source_governance_conservative,OK,[],nenhuma célula com LSTM como fonte primária (c...,Promoção de LSTM exige recomendação governada ...
5,primary_summary_rows,OK,8,>= 8,Esperado pelo menos um resumo primário por cél...



Resumo primário por célula:


,cell_id,scenario_label,score_source,score_label,is_primary_score_source,score_source_file,model,feature_set,n_scored_rows,n_total_episodes,duration_one_window_pct,tau_reference,precision_tau_reference,recall_tau_reference,f1_tau_reference,fpr_tau_reference,fnr_tau_reference,false_alerts_per_day_tau_reference,positive_rate_tau_reference,alert_rate_tau_reference,n_evaluable_episodes_tau_reference,n_anticipated_episodes_tau_reference,episode_anticipation_rate_tau_reference,lead_time_minutes_mean_tau_reference,lead_time_minutes_median_tau_reference,tau_opt_cFP1_cFN10,cost_opt_cFP1_cFN10,recall_at_opt_cFP1_cFN10,fpr_at_opt_cFP1_cFN10,fnr_at_opt_cFP1_cFN10,tau_opt_cFP1_cFN20,cost_opt_cFP1_cFN20,recall_at_opt_cFP1_cFN20,fpr_at_opt_cFP1_cFN20,fnr_at_opt_cFP1_cFN20,tau_opt_cFP1_cFN5,cost_opt_cFP1_cFN5,recall_at_opt_cFP1_cFN5,fpr_at_opt_cFP1_cFN5,fnr_at_opt_cFP1_cFN5
0,a,cell_a_W5_K24_H12_train_m2s_P1,NB11_FULL,cell_a_W5_K24_H12_train_m2s_P1 | NB11_FULL,1,/content/drive/MyDrive/Mestrado/04-reports/99_...,logistic_regression,temporal_core,1492,191,0.560209,0.5,0.530612,0.180556,0.269430,0.017062,0.819444,4.439678,0.096515,0.032842,11,3,0.272727,38.333333,30.0,0.01,0.999258,1.0,0.999258,0.0,0.01,0.999258,1.0,0.999258,0.0,0.05,0.877761,0.972222,0.738872,0.027778
3,b,cell_b_W5_K24_H12_train_m2s_P1,NB11_FULL,cell_b_W5_K24_H12_train_m2s_P1 | NB11_FULL,1,/content/drive/MyDrive/Mestrado/04-reports/99_...,hist_gradient_boosting,temporal_core,1451,219,0.611872,0.5,0.592593,0.435374,0.501961,0.033742,0.564626,8.733287,0.101309,0.074431,10,4,0.400000,53.750000,60.0,0.00,1.000000,1.0,1.000000,0.0,0.00,1.000000,1.0,1.000000,0.0,0.00,1.000000,1.000000,1.000000,0.000000
6,c,cell_c_W5_K24_H6_train_m2s_P1,NB11_FULL,cell_c_W5_K24_H6_train_m2s_P1 | NB11_FULL,1,/content/drive/MyDrive/Mestrado/04-reports/99_...,hist_gradient_boosting,temporal_core,1458,178,0.533708,0.5,0.735294,0.260417,0.384615,0.006608,0.739583,1.777778,0.065844,0.023320,13,5,0.384615,13.000000,10.0,0.00,1.000000,1.0,1.000000,0.0,0.00,1.000000,1.0,1.000000,0.0,0.00,1.000000,1.000000,1.000000,0.000000
9,d,cell_d_W5_K24_H12_train_m2s_P2,NB11_FULL,cell_d_W5_K24_H12_train_m2s_P2 | NB11_FULL,1,/content/drive/MyDrive/Mestrado/04-reports/99_...,logistic_regression,raw_7,1745,74,0.000000,0.5,0.666667,0.500000,0.571429,0.001731,0.500000,0.495129,0.006877,0.005158,1,1,1.000000,35.000000,35.0,0.00,1.000000,1.0,1.000000,0.0,0.00,1.000000,1.0,1.000000,0.0,0.00,1.000000,1.000000,1.000000,0.000000
10,e,cell_e_W5_K12_H6_train_m2s_P1,NB11_FULL,cell_e_W5_K12_H6_train_m2s_P1 | NB11_FULL,1,/content/drive/MyDrive/Mestrado/04-reports/99_...,hist_gradient_boosting,temporal_core,1658,110,0.663636,0.5,0.475000,0.250000,0.327586,0.013274,0.750000,3.647768,0.045838,0.024125,9,3,0.333333,50.000000,45.0,0.00,1.000000,1.0,1.000000,0.0,0.00,1.000000,1.0,1.000000,0.0,0.00,1.000000,1.000000,1.000000,0.000000
13,f,cell_f_W5_K24_H12_train_m2s_P1,NB11_FULL,cell_f_W5_K24_H12_train_m2s_P1 | NB11_FULL,1,/content/drive/MyDrive/Mestrado/04-reports/99_...,logistic_regression,temporal_core,1424,135,0.644444,0.5,0.826087,0.164502,0.274368,0.006706,0.835498,1.617978,0.162219,0.032303,16,1,0.062500,60.000000,60.0,0.06,0.981559,1.0,0.981559,0.0,0.06,0.981559,1.0,0.981559,0.0,0.06,0.981559,1.000000,0.981559,0.000000
16,g,cell_g_W5_K24_H12_train_m2s_P1,NB11_FULL,cell_g_W5_K24_H12_train_m2s_P1 | NB11_FULL,1,/content/drive/MyDrive/Mestrado/04-reports/99_...,hist_gradient_boosting,temporal_core,1155,293,0.709898,0.5,0.631300,0.485714,0.549020,0.209023,0.514286,34.659740,0.424242,0.326407,30,18,0.600000,50.277778,60.0,0.00,1.000000,1.0,1.000000,0.0,0.00,1.000000,1.0,1.000000,0.0,0.00,1.000000,1.000000,1.000000,0.000000
19,h,cell_h_W5_K24_H12_train_m2s_P1,NB11_FULL,cell_h_W5_K24_H12_train_m2s_P1 | NB11_FULL,1,/content/drive/MyDrive/Mestrado/04-reports/99_...,logistic_regression,temporal_core,1296,225,0.786667,0.5,0.743750,0.309896,0.437500,0.044956,0.690104,9.111111,0.296296,0.123457,25,5,0.200000,53.000000,60.0,0.09,0.992325,1.0,0.992325,0.


Taus ótimos por célula/cenário/fonte/custo:


,cell_id,scenario_label,score_source,score_label,tau,cost_label,c_fp,c_fn,cost,cost_normalized,fpr,fnr,precision,recall,f1,false_alerts_per_day,model,feature_set,optimality_rule
0,a,cell_a_W5_K24_H12_train_m2s_P1,NB11_FULL,cell_a_W5_K24_H12_train_m2s_P1 | NB11_FULL,0.01,cFP1_cFN10,1.0,10.0,0.999258,0.090842,0.999258,0.000000,0.096579,1.000000,0.176147,260.010724,logistic_regression,temporal_core,minimize_cFP_FPR_plus_cFN_FNR_then_low_FNR_low...
1,a,cell_a_W5_K24_H12_train_m2s_P1,NB11_FULL,cell_a_W5_K24_H12_train_m2s_P1 | NB11_FULL,0.01,cFP1_cFN20,1.0,20.0,0.999258,0.047584,0.999258,0.000000,0.096579,1.000000,0.176147,260.010724,logistic_regression,temporal_core,minimize_cFP_FPR_plus_cFN_FNR_then_low_FNR_low...
2,a,cell_a_W5_K24_H12_train_m2s_P1,NB11_FULL,cell_a_W5_K24_H12_train_m2s_P1 | NB11_FULL,0.05,cFP1_cFN5,1.0,5.0,0.877761,0.146294,0.738872,0.027778,0.123239,0.972222,0.218750,192.257373,logistic_regression,temporal_core,minimize_cFP_FPR_plus_cFN_FNR_then_low_FNR_low...
3,a,cell_a_W5_K24_H12_train_m2s_P1,NB13A_FULL,cell_a_W5_K24_H12_train_m2s_P1 | NB13A_FULL,0.17,cFP1_cFN10,1.0,10.0,0.958457,0.087132,0.958457,0.000000,0.100279,1.000000,0.182278,249.394102,unknown,winner_temporal_core,minimize_cFP_FPR_plus_cFN_FNR_then_low_FNR_low...
4,a,cell_a_W5_K24_H12_train_m2s_P1,NB13A_FULL,cell_a_W5_K24_H12_train_m2s_P1 | NB13A_FULL,0.17,cFP1_cFN20,1.0,20.0,0.958457,0.045641,0.958457,0.000000,0.100279,1.000000,0.182278,249.394102,unknown,winner_temporal_core,minimize_cFP_FPR_plus_cFN_FNR_then_low_FNR_low...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,h,cell_h_W5_K24_H12_train_m2s_P1,NB13A_FULL,cell_h_W5_K24_H12_train_m2s_P1 | NB13A_FULL,0.00,cFP1_cFN20,1.0,20.0,1.000000,0.047619,1.000000,0.000000,0.296296,1.000000,0.457143,202.666667,unknown,winner_temporal_core,minimize_cFP_FPR_plus_cFN_FNR_then_low_FNR_low...
62,h,cell_h_W5_K24_H12_train_m2s_P1,NB13A_FULL,cell_h_W5_K24_H12_train_m2s_P1 | NB13A_FULL,0.00,cFP1_cFN5,1.0,5.0,1.000000,0.166667,1.000000,0.000000,0.296296,1.000000,0.457143,202.666667,unknown,winner_temporal_core,minimize_cFP_FPR_plus_cFN_FNR_then_low_FNR_low...
63,h,cell_h_W5_K24_H12_train_m2s_P1,NB13_FULL,cell_h_W5_K24_H12_train_m2s_P1 | NB13_FULL,0.29,cFP1_cFN10,1.0,10.0,0.995614,0.090510,0.995614,0.000000,0.297214,1.000000,0.458234,201.777778,unknown,winner_temporal_core,minimize_cFP_FPR_plus_cFN_FNR_then_low_FNR_low...
64,h,cell_h_W5_K24_H12_train_m2s_P1,NB13_FULL,cell_h_W5_K24_H12_train_m2s_P1 | NB13_FULL,0.29,cFP1_cFN20,1.0,20.0,0.995614,0.047410,0.995614,0.000000,0.297214,1.000000,0.458234,201.777778,unknown,winner_temporal_core,minimize_cFP_FPR_plus_cFN_FNR_then_low_FNR_low...



Artefatos aggregate:


,artifact,path,exists
0,scores_selected_allcells,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
1,threshold_metrics_allcells,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
2,cost_curve_allcells,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
3,optimal_tau_allcells,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
4,false_alerts_allcells,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
5,episode_anticipability_by_tau_allcells,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
6,episode_anticipability_allcells,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
7,episode_duration_summary_allcells,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
8,scenario_summary_by_cell,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
9,cost_summary_by_cell,/content/drive/MyDrive/Mestrado/04-reports/99_...,True



Todas as células processadas concluíram sem erro registrado.

Leitura preliminar: o NB14_FULL converteu escores por célula em análise de limiar tau, custo C(tau), falsos alertas por dia, antecipabilidade de episódios e lead time. A integração com NB13a_FULL respeita nb14_score_source_recommendation. O NB15_FULL deve consumir prioritariamente 14_FULL_scenario_summary_by_cell.csv, 14_FULL_cost_summary_by_cell.csv, 14_FULL_optimal_tau_by_cost_allcells.csv e as figuras aggregate.


# NB14_FULL — Conclusão da etapa

## 8. Conclusão da etapa

A execução do `NB14_FULL — Análise de decisão por célula` foi concluída com sucesso para as oito células Borg (`a`–`h`). A etapa consumiu os escores já produzidos no ramo `_FULL`, preservou a independência experimental por `cell_id` e consolidou os resultados apenas em artefatos tabulares e figuras `aggregate`. O notebook não reabriu treinamento, tuning, seleção de modelo, cálculo de limiar, split ou rotulagem; episódios, estados e alvo foram herdados dos notebooks upstream.

### 8.1 Resultado operacional

| Item                                               | Resultado                                                                                                                             |
| -------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------- |
| Células solicitadas                                | a, b, c, d, e, f, g, h                                                                                                                |
| Células concluídas                                 | a, b, c, d, e, f, g, h (8/8)                                                                                                          |
| Células com erro                                   | nenhuma; `14_FULL_execution_errors.csv` vazio                                                                                         |
| Fonte primária predominante de escores             | NB11_FULL (8/8)                                                                                                                       |
| Cenário principal predominante                     | W5_K24_H12_train_m2s_P1 (5/8)                                                                                                         |
| Células com sensibilidade NB12/NB13/NB13a incluída | a, b, c, e, f, g, h (NB13_FULL e NB13a_FULL); célula d sem sensibilidade LSTM; NB12 não presente nos artefatos consumidos.            |
| `pilot_validation_summary` OK/WARN/FAIL            | OK=6; WARN=0; FAIL=0                                                                                                                  |
| Governança NB13/NB13a→NB14                         | `score_source_governance_conservative = OK`; nenhuma célula com LSTM como fonte primária                                              |
| Artefatos aggregate gerados                        | 16 arquivos em `aggregate/` + 3 figuras em `aggregate/figures/`; manifesto com 146 artefatos hasheados; 147 arquivos totais na árvore |

### 8.2 Leitura por célula

| cell_id | score primário                                     | episódios totais | episódios avaliáveis | episódios antecipados | taxa antecipação tau=0,50 | lead time mediano | F1 tau=0,50 | falsos alertas/dia | tau ótimo 1:10 |
| ------- | -------------------------------------------------- | ---------------: | -------------------: | --------------------: | ------------------------: | ----------------: | ----------: | -----------------: | -------------: |
| a       | NB11_FULL / logistic_regression / temporal_core    |              191 |                   11 |                     3 |                     27,3% |          30,0 min |       0,269 |               4,44 |           0,01 |
| b       | NB11_FULL / hist_gradient_boosting / temporal_core |              219 |                   10 |                     4 |                     40,0% |          60,0 min |       0,502 |               8,73 |           0,00 |
| c       | NB11_FULL / hist_gradient_boosting / temporal_core |              178 |                   13 |                     5 |                     38,5% |          10,0 min |       0,385 |               1,78 |           0,00 |
| d       | NB11_FULL / logistic_regression / raw_7            |               74 |                    1 |                     1 |                    100,0% |          35,0 min |       0,571 |               0,50 |           0,00 |
| e       | NB11_FULL / hist_gradient_boosting / temporal_core |              110 |                    9 |                     3 |                     33,3% |          45,0 min |       0,328 |               3,65 |           0,00 |
| f       | NB11_FULL / logistic_regression / temporal_core    |              135 |                   16 |                     1 |                      6,2% |          60,0 min |       0,274 |               1,62 |           0,06 |
| g       | NB11_FULL / hist_gradient_boosting / temporal_core |              293 |                   30 |                    18 |                     60,0% |          60,0 min |       0,549 |              34,66 |           0,00 |
| h       | NB11_FULL / logistic_regression / temporal_core    |              225 |                   25 |                     5 |                     20,0% |          60,0 min |       0,437 |               9,11 |           0,09 |

### 8.3 Consolidação dos resultados principais

No limiar de referência `tau = 0,50`, a análise primária avaliou 11.679 linhas pontuadas e 1.425 episódios totais nas oito células. A soma tabular por célula indica 115 episódios avaliáveis, dos quais 40 foram antecipados, resultando em taxa global descritiva de 34,8%. Essa taxa é apenas uma consolidação por célula; ela não deve ser interpretada como métrica calculada sobre uma série temporal única `allcells`.

O desempenho por janela apresentou heterogeneidade relevante: o `F1` em `tau = 0,50` variou de 0,269 (`cell_a`) a 0,571 (`cell_d`), enquanto os falsos alertas por dia variaram de 0,50 (`cell_d`) a 34,66 (`cell_g`). A taxa de antecipação de episódios também foi heterogênea: `cell_g` antecipou 18 de 30 episódios avaliáveis (60,0%), enquanto `cell_f` antecipou 1 de 16 (6,3%). A `cell_d` aparece com 100,0% de antecipação, mas com apenas 1 episódio avaliável, devendo ser tratada como leitura descritiva de baixa contagem.

Os limiares ótimos por custo para a relação `1:10` ficaram frequentemente baixos, com `tau = 0,00` nas células `b`, `c`, `d`, `e` e `g`; `tau = 0,01` na célula `a`; `tau = 0,06` na célula `f`; e `tau = 0,09` na célula `h`. Esse comportamento é coerente com a função de custo quando falsos negativos são fortemente penalizados: a minimização de `C(tau)` tende a priorizar recall, mesmo à custa de aumento de alertas. Portanto, esses limiares devem ser tratados como análise de trade-off decisório, não como recomendação operacional direta sem calibração de custos e capacidade de triagem.

### 8.4 Interpretação metodológica

O `NB14_FULL` deve ser lido como a etapa de passagem entre desempenho preditivo e decisão operacional. A métrica por janela informa o comportamento do escore sob diferentes limiares `tau`; a métrica por episódio informa se a política de alerta conseguiria antecipar episódios críticos dentro do horizonte herdado `H`; e a função `C(tau)` registra o compromisso entre falsos positivos e falsos negativos para diferentes assimetrias de custo.

A leitura `aggregate` preserva a independência das células. Diferenças entre `cell_id` são evidência de heterogeneidade operacional do Borg completo, não ruído a ser eliminado por concatenação. Células com poucos episódios avaliáveis ou muitos episódios de duração mínima devem ser tratadas com linguagem descritiva, principalmente quando houver baixa contagem nas faixas de duração.

A presença de cenários principais distintos por célula também deve ser preservada na interpretação. O cenário predominante foi `W5_K24_H12_train_m2s_P1` em cinco células (`a`, `b`, `f`, `g`, `h`), mas `cell_c`, `cell_d` e `cell_e` herdaram cenários vencedores distintos do upstream. Isso é coerente com a estratégia _FULL: a célula é a réplica experimental, e não uma partição decorativa de uma série única.

### 8.5 Decisão para o próximo notebook

O `NB14_FULL` está aprovado para alimentar o `NB15_FULL`. O próximo notebook deve consumir prioritariamente:

* `14_FULL_scenario_summary_by_cell.csv`;
* `14_FULL_cost_summary_by_cell.csv`;
* `14_FULL_optimal_tau_by_cost_allcells.csv`;
* `14_FULL_episode_anticipability_allcells.csv`;
* `14_FULL_episode_anticipability_by_tau_allcells.csv`;
* `14_FULL_false_alerts_by_tau_allcells.csv`;
* figuras em `14_FULL_decision_analysis/aggregate/figures/`.

Antes de usar qualquer número na dissertação, deve-se conferir se `14_FULL_pilot_validation_summary.csv` permanece sem `FAIL` e se `14_FULL_execution_errors.csv` continua vazio. Nesta execução, ambos os critérios foram atendidos.
